# CoT-FD6 reproducibility notebook

**Associated manuscript:**  
*Explainable LLM-Assisted Fault Diagnosis for Smart Manufacturing Using Multi-Trace Diagnostic Rationales and Verification*

This notebook is the cleaned publication version of the experimental workflow used for the paper. It preserves the reported experimental configuration while removing exploratory smoke/pilot cells, personal Google Drive paths, duplicate figure code, and legacy verifier-selection logic that is not part of the final methodology.

**Important interpretation boundary:** generated rationales are explicit model outputs. They are not treated as access to hidden chain-of-thought.

**Safety:** expensive OpenAI API calls are disabled by default. See `docs/REPRODUCIBILITY.md` before enabling them.

## 1. Environment

Create a Python 3.12 environment and install the repository dependencies:

```bash
pip install -r requirements.txt
```

For Google Colab, clone/open the repository first, then install the same requirements. Do not store an API key in the notebook.

In [ ]:
from __future__ import annotations

import os
import re
import json
import math
import time
import random
import hashlib
import warnings
from pathlib import Path
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# Repository-relative paths
# ------------------------------------------------------------------
def _find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    return start

ROOT = Path(os.getenv("COTFD6_ROOT", str(_find_repo_root()))).resolve()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
PROMPTS = ROOT / "prompts"
FIGURES = RESULTS / "figures"

for p in (DATA, RESULTS, PROMPTS, FIGURES):
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Frozen experimental configuration used for the paper
# ------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL = "gpt-4.1-mini-2025-04-14"
TEMPERATURE = 0.70
TOP_P = 1.0
MAX_OUTPUT_TOKENS = 600

# Safety guard: expensive API experiments are OFF by default.
# To rerun them, set COTFD6_RUN_LLM=1 before starting the kernel.
RUN_LLM_EXPERIMENTS = os.getenv("COTFD6_RUN_LLM", "0").strip().lower() in {
    "1", "true", "yes", "on"
}

# Mock mode is only for pipeline checks; mock outputs must never be reported.
MOCK_MODE = os.getenv("COTFD6_MOCK_MODE", "0").strip().lower() in {
    "1", "true", "yes", "on"
}

# Historical standard-rate token prices used to estimate the reported cost.
# These are frozen experiment metadata, not claims about current API pricing.
PRICE_INPUT_PER_1M_USD = 0.40
PRICE_OUTPUT_PER_1M_USD = 1.60

TEST_SIZE = 0.25
SPLIT_SEED = 42

MAIN_N = 5
MAIN_REPEATS = 5
LLM_MAX = 100

TRACE_NS = (3, 5, 10)
TRACE_REPEATS = 3
TRACE_MAX = 24

ABLATION_REPEATS = 3
ABLATION_MAX = 36

ALPHA = 0.50
PROTOTYPE_TAU = 1.0
DIRECTION_EPS = 0.25
MEMORY_K = 3

# Synthetic data
SYNTH_RUNS_PER_CLASS = 50
SYNTH_LEN = 600
SYNTH_ONSET = 300
SYNTH_WINDOW = 120
SYNTH_STRIDE = 60

# NASA C-MAPSS FD001
NASA_WINDOW = 50
NASA_STRIDE = 10
RUL_THRESHOLD = 30
NASA_PATH = Path(
    os.getenv("COTFD6_NASA_PATH", str(DATA / "train_FD001.txt"))
).resolve()

CONFIG = {
    "MODEL": MODEL,
    "TEMPERATURE": TEMPERATURE,
    "TOP_P": TOP_P,
    "MAX_OUTPUT_TOKENS": MAX_OUTPUT_TOKENS,
    "MOCK_MODE": MOCK_MODE,
    "TEST_SIZE": TEST_SIZE,
    "SPLIT_SEED": SPLIT_SEED,
    "MAIN_N": MAIN_N,
    "MAIN_REPEATS": MAIN_REPEATS,
    "LLM_MAX": LLM_MAX,
    "TRACE_NS": list(TRACE_NS),
    "TRACE_REPEATS": TRACE_REPEATS,
    "TRACE_MAX": TRACE_MAX,
    "TRACE_SENSITIVITY_MEMORY": True,
    "ABLATION_REPEATS": ABLATION_REPEATS,
    "ABLATION_MAX": ABLATION_MAX,
    "ALPHA": ALPHA,
    "PROTOTYPE_TAU": PROTOTYPE_TAU,
    "DIRECTION_EPS": DIRECTION_EPS,
    "MEMORY_K": MEMORY_K,
    "SYNTH_RUNS_PER_CLASS": SYNTH_RUNS_PER_CLASS,
    "SYNTH_LEN": SYNTH_LEN,
    "SYNTH_ONSET": SYNTH_ONSET,
    "SYNTH_WINDOW": SYNTH_WINDOW,
    "SYNTH_STRIDE": SYNTH_STRIDE,
    "NASA_WINDOW": NASA_WINDOW,
    "NASA_STRIDE": NASA_STRIDE,
    "RUL_THRESHOLD": RUL_THRESHOLD,
    "PRICE_INPUT_PER_1M_USD": PRICE_INPUT_PER_1M_USD,
    "PRICE_OUTPUT_PER_1M_USD": PRICE_OUTPUT_PER_1M_USD,
}

(RESULTS / "experiment_config.json").write_text(
    json.dumps(CONFIG, indent=2),
    encoding="utf-8",
)

print("Repository root:", ROOT)
print("NASA FD001 path:", NASA_PATH)
print("NASA file present:", NASA_PATH.exists())
print("Expensive LLM experiments enabled:", RUN_LLM_EXPERIMENTS)
print("Mock mode:", MOCK_MODE)

## 2. OpenAI client and structured-output utilities

In [ ]:
# OpenAI client + robust structured-output/usage logger

API_LOGS = []

def cost_usd(input_tokens: int, output_tokens: int) -> float:
    return (
        input_tokens / 1e6 * PRICE_INPUT_PER_1M_USD
        + output_tokens / 1e6 * PRICE_OUTPUT_PER_1M_USD
    )

def parse_json(text: str) -> dict:
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    first, last = text.find("{"), text.rfind("}")
    if first >= 0 and last > first:
        text = text[first:last + 1]
    try:
        return json.loads(text)
    except Exception:
        return {
            "diagnosis": "unknown",
            "alternative": None,
            "evidence": [],
            "contradictions": [],
            "rationale": text[:500],
            "_parse_error": True,
        }

def mock_json(labels, features):
    score = float(np.mean(np.abs(list(features.values()))))
    if labels == ["healthy", "degraded"]:
        label = "degraded" if score > 0.65 else "healthy"
    else:
        label = random.choice(labels)

    evidence = []
    for feature, value in sorted(
        features.items(), key=lambda kv: abs(kv[1]), reverse=True
    )[:3]:
        direction = (
            "positive" if value > DIRECTION_EPS
            else "negative" if value < -DIRECTION_EPS
            else "near_zero"
        )
        evidence.append({
            "feature": feature,
            "direction": direction,
            "importance": "high",
        })

    return {
        "diagnosis": label,
        "alternative": None,
        "evidence": evidence,
        "contradictions": [],
        "rationale": "MOCK pipeline test only.",
    }

client = None
if not MOCK_MODE and RUN_LLM_EXPERIMENTS:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is required when COTFD6_RUN_LLM=1. "
            "Do not place API keys directly in this notebook."
        )
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def llm_call(system, user, meta, retries=3):
    if MOCK_MODE:
        parsed = mock_json(meta["labels"], meta["features"])
        usage = {
            "endpoint": "mock",
            "input_tokens": 0,
            "output_tokens": 0,
            "latency_s": 0.0,
            "cost_usd": 0.0,
            "success": True,
        }
        return parsed, usage

    if not RUN_LLM_EXPERIMENTS or client is None:
        raise RuntimeError(
            "LLM calls are disabled. Set COTFD6_RUN_LLM=1 and provide "
            "OPENAI_API_KEY before starting the kernel to rerun API experiments."
        )

    last_error = None
    for attempt in range(retries):
        started = time.perf_counter()
        try:
            try:
                response = client.responses.create(
                    model=MODEL,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    input=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ],
                )
                text = response.output_text
                usage_obj = getattr(response, "usage", None)
                endpoint = "responses"
                input_tokens = int(
                    getattr(usage_obj, "input_tokens", 0) or 0
                )
                output_tokens = int(
                    getattr(usage_obj, "output_tokens", 0) or 0
                )
            except Exception:
                response = client.chat.completions.create(
                    model=MODEL,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    max_tokens=MAX_OUTPUT_TOKENS,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ],
                )
                text = response.choices[0].message.content
                usage_obj = getattr(response, "usage", None)
                endpoint = "chat.completions"
                input_tokens = int(
                    getattr(usage_obj, "prompt_tokens", 0) or 0
                )
                output_tokens = int(
                    getattr(usage_obj, "completion_tokens", 0) or 0
                )

            usage = {
                "endpoint": endpoint,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_s": time.perf_counter() - started,
                "cost_usd": cost_usd(input_tokens, output_tokens),
                "success": True,
            }
            API_LOGS.append({
                **{
                    k: v
                    for k, v in meta.items()
                    if k not in ("features", "labels")
                },
                **usage,
            })
            return parse_json(text), usage

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(2 ** attempt)

    usage = {
        "endpoint": "failed",
        "input_tokens": 0,
        "output_tokens": 0,
        "latency_s": np.nan,
        "cost_usd": np.nan,
        "success": False,
        "error": last_error,
    }
    API_LOGS.append({
        **{
            k: v
            for k, v in meta.items()
            if k not in ("features", "labels")
        },
        **usage,
    })
    return {
        "diagnosis": "unknown",
        "alternative": None,
        "evidence": [],
        "contradictions": [],
        "rationale": last_error,
    }, usage

## 3. Feature extraction and datasets

In [ ]:
# 3. Common window features
def extract_features(w,sensors):
    out={}; n=len(w); third=max(1,n//3); x=np.linspace(0,1,n)
    for c in sensors:
        v=w[c].astype(float).to_numpy(); sd=np.std(v)
        out[f"{c}_delta_mean"]=float(v[-third:].mean()-v[:third].mean())
        out[f"{c}_z_last"]=float((v[-1]-v.mean())/(sd+1e-8))
        out[f"{c}_slope"]=float(np.polyfit(x,v,1)[0])
    return out

def fcols(df): return [c for c in df.columns if c.endswith(("_delta_mean","_z_last","_slope"))]

def strat_cap(df,n,seed=42):
    if len(df)<=n:return df.copy().reset_index(drop=True)
    idx=np.arange(len(df)); keep,_=train_test_split(idx,train_size=n,random_state=seed,stratify=df.label)
    return df.iloc[np.sort(keep)].reset_index(drop=True)

In [ ]:
# 4. Synthetic controlled sandbox
SYNTH_LABELS=["winding_short","bearing_friction","overload","sensor_drift","overvoltage"]

def simulate_motor(n=600,fault="winding_short",seed=0):
    rng=np.random.default_rng(seed); t=np.arange(n)
    I=10+.3*np.sin(2*np.pi*t/120)+rng.normal(0,.1,n)
    T=40+.5*np.sin(2*np.pi*t/200+.4)+rng.normal(0,.2,n)
    V=.2+.02*np.sin(2*np.pi*t/60)+rng.normal(0,.01,n); s=n//2
    if fault=="winding_short": I[s:]+=np.linspace(0,4,n-s); T[s:]+=np.linspace(0,12,n-s)
    elif fault=="bearing_friction": V[s:]+=np.linspace(0,.25,n-s); T[s:]+=np.linspace(0,8,n-s); I[s:]+=np.linspace(0,.8,n-s)
    elif fault=="overload": I[s:]+=np.linspace(0,2,n-s); T[s:]+=np.linspace(0,7,n-s); V[s:]+=np.linspace(0,.05,n-s)
    elif fault=="sensor_drift": T+=np.linspace(0,10,n)
    elif fault=="overvoltage":
        I[s:]+=np.linspace(0,3.5,n-s); V[s:]+=np.linspace(0,.06,n-s); d=s+n//10; T[d:]+=np.linspace(0,6,n-d)
    return pd.DataFrame({"time":t,"current":I,"temperature":T,"vibration":V})

def build_synth():
    rows=[]; sensors=["current","temperature","vibration"]
    for li,lab in enumerate(SYNTH_LABELS):
        for r in range(SYNTH_RUNS_PER_CLASS):
            rid=f"{lab}_{r:03d}"; ser=simulate_motor(SYNTH_LEN,lab,10000*li+r+SEED)
            for st in range(SYNTH_ONSET,SYNTH_LEN-SYNTH_WINDOW+1,SYNTH_STRIDE):
                rows.append({"dataset":"synthetic","instance_id":f"{rid}_w{st}","group_id":rid,
                             "window_start":st,"window_end":st+SYNTH_WINDOW-1,"label":lab,
                             **extract_features(ser.iloc[st:st+SYNTH_WINDOW],sensors)})
    return pd.DataFrame(rows)

synth=build_synth()
print(synth.shape); print(synth.label.value_counts())

In [ ]:
# 5. NASA FD001: true windowing + explicit RUL labels
def load_fd001(path):
    cols=["unit","time"]+[f"op{i}" for i in range(1,4)]+[f"s{i}" for i in range(1,22)]
    d=pd.read_csv(path,sep=r"\s+",header=None,names=cols,engine="python").dropna(axis=1,how="all")
    d["unit"]=d.unit.astype(int); d["time"]=d.time.astype(int); return d

def build_nasa(raw):
    rows=[]; sensors=[f"s{i}" for i in range(1,22)]; mx=raw.groupby("unit").time.max().to_dict()
    for unit,u in raw.groupby("unit"):
        u=u.sort_values("time").reset_index(drop=True)
        for i in range(0,len(u)-NASA_WINDOW+1,NASA_STRIDE):
            w=u.iloc[i:i+NASA_WINDOW]; end=int(w.time.iloc[-1]); rul=int(mx[int(unit)]-end)
            rows.append({"dataset":"nasa_fd001","instance_id":f"u{int(unit):03d}_e{end:04d}","group_id":int(unit),
                         "window_start":int(w.time.iloc[0]),"window_end":end,"rul_end":rul,
                         "label":"degraded" if rul<=RUL_THRESHOLD else "healthy",
                         **extract_features(w,sensors)})
    return pd.DataFrame(rows)

if NASA_PATH.exists():
    nasa_raw=load_fd001(NASA_PATH); nasa=build_nasa(nasa_raw)
    print(nasa.shape); print(nasa.label.value_counts())
else:
    nasa=None
    print("Upload/place train_FD001.txt at:",NASA_PATH)

NASA C-MAPSS FD001 is not redistributed by this repository. Download `train_FD001.txt` from the NASA Prognostics Data Repository and place it in `data/`, or set `COTFD6_NASA_PATH` to its location.

In [ ]:
# ============================================================
# 6. Leakage-safe train/test split + fixed LLM evaluation subset
# ============================================================
#
# DESIGN PRINCIPLES
#
# SYNTHETIC:
#   - Split at SIMULATION-RUN level (group_id)
#   - Stratify by fault class
#   - No windows from the same simulated run may appear
#     in both training and test sets
#
# NASA FD001:
#   - Split strictly at ENGINE level (group_id = unit)
#   - 75% engines train / 25% engines test
#   - No engine appears in both train and test
#
# BOTH:
#   - Feature selection/scaling learned from TRAINING data only
#   - Fixed stratified subset of up to LLM_MAX test windows
#     is created for the expensive CoT-FD6 experiments
# ============================================================


@dataclass
class DS:
    name: str
    labels: list
    train: pd.DataFrame
    test: pd.DataFrame
    llm_test: pd.DataFrame
    features: list
    scaler: StandardScaler


def prepare(df, name, labels):

    # ========================================================
    # A. SYNTHETIC DATASET
    # Stratified split at SIMULATION-RUN level
    # ========================================================
    if name == "synthetic":

        # Each group_id represents one complete simulated run.
        # Each run belongs to exactly one fault class.
        groups = (
            df[["group_id", "label"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )

        print(
            f"\n[{name}] Total simulation runs:",
            len(groups)
        )

        print(
            f"[{name}] Runs per class before split:"
        )
        print(
            groups["label"]
            .value_counts()
            .sort_index()
        )

        # Stratify by fault class at RUN level.
        train_groups, test_groups = train_test_split(
            groups,
            test_size=TEST_SIZE,
            random_state=SPLIT_SEED,
            stratify=groups["label"]
        )

        train_ids = set(
            train_groups["group_id"]
        )

        test_ids = set(
            test_groups["group_id"]
        )

        # Recover all windows belonging to the selected runs.
        train = (
            df[
                df["group_id"].isin(train_ids)
            ]
            .copy()
            .reset_index(drop=True)
        )

        test = (
            df[
                df["group_id"].isin(test_ids)
            ]
            .copy()
            .reset_index(drop=True)
        )

    # ========================================================
    # B. NASA FD001
    # Strict ENGINE-WISE split
    # ========================================================
    elif name == "nasa_fd001":

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=TEST_SIZE,
            random_state=SPLIT_SEED
        )

        train_idx, test_idx = next(
            splitter.split(
                df,
                groups=df["group_id"]
            )
        )

        train = (
            df.iloc[train_idx]
            .copy()
            .reset_index(drop=True)
        )

        test = (
            df.iloc[test_idx]
            .copy()
            .reset_index(drop=True)
        )

    else:
        raise ValueError(
            f"Unknown dataset name: {name}"
        )

    # ========================================================
    # C. Leakage checks
    # ========================================================

    train_group_set = set(
        train["group_id"]
    )

    test_group_set = set(
        test["group_id"]
    )

    overlap = (
        train_group_set
        .intersection(test_group_set)
    )

    assert len(overlap) == 0, (
        f"DATA LEAKAGE DETECTED in {name}: "
        f"{len(overlap)} groups appear in both train and test."
    )

    # ========================================================
    # D. Feature selection
    #
    # Identify candidate feature columns, then remove
    # near-constant features using TRAINING data only.
    # ========================================================

    all_features = fcols(df)

    keep = [
        f
        for f in all_features
        if float(train[f].std()) > 1e-12
    ]

    dropped = sorted(
        set(all_features) - set(keep)
    )

    if not keep:
        raise ValueError(
            f"No usable features remained for dataset {name}."
        )

    # ========================================================
    # E. Scaling
    #
    # IMPORTANT:
    # StandardScaler is fitted ONLY on training data.
    # Test-set information is therefore not used.
    # ========================================================

    scaler = StandardScaler()

    scaler.fit(
        train[keep]
    )

    # ========================================================
    # F. Fixed LLM comparison subset
    #
    # This subset comes ONLY from the held-out test partition.
    # strat_cap() preserves the class distribution.
    #
    # For NASA this previously produced:
    #   Healthy  = 80
    #   Degraded = 20
    # across all 25 held-out engines.
    # ========================================================

    llm_test = strat_cap(
        test,
        LLM_MAX,
        SPLIT_SEED
    )

    # ========================================================
    # G. Final integrity checks
    # ========================================================

    assert set(
        llm_test["instance_id"]
    ).issubset(
        set(test["instance_id"])
    ), (
        f"{name}: LLM subset contains instances "
        f"outside the held-out test set."
    )

    assert set(
        llm_test["group_id"]
    ).issubset(
        test_group_set
    ), (
        f"{name}: LLM subset contains groups "
        f"outside the test partition."
    )

    return DS(
        name=name,
        labels=labels,
        train=train,
        test=test,
        llm_test=llm_test,
        features=keep,
        scaler=scaler
    )


# ============================================================
# 6.1 Prepare both datasets
# ============================================================

PREP = {}

# Synthetic dataset
PREP["synthetic"] = prepare(
    synth,
    "synthetic",
    SYNTH_LABELS
)

# NASA dataset — only add if train_FD001.txt was successfully loaded
if nasa is not None:

    PREP["nasa_fd001"] = prepare(
        nasa,
        "nasa_fd001",
        ["healthy", "degraded"]
    )


# ============================================================
# 6.2 Diagnostic summary
# ============================================================

for name, d in PREP.items():

    print("\n" + "=" * 70)
    print(
        f"DATASET: {name}"
    )
    print("=" * 70)

    print(
        f"Train windows : {len(d.train)}"
    )

    print(
        f"Test windows  : {len(d.test)}"
    )

    print(
        f"LLM subset    : {len(d.llm_test)}"
    )

    print(
        f"Train groups  : "
        f"{d.train['group_id'].nunique()}"
    )

    print(
        f"Test groups   : "
        f"{d.test['group_id'].nunique()}"
    )

    print(
        f"Features used : {len(d.features)}"
    )

    # --------------------------------------------------------
    # Test-set class distribution
    # --------------------------------------------------------

    print(
        "\nTest-set class distribution:"
    )

    print(
        d["test"]["label"].value_counts()
        if isinstance(d, dict)
        else d.test["label"].value_counts()
    )

    print(
        "\nTest-set class percentages:"
    )

    print(
        (
            d.test["label"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        )
    )

    # --------------------------------------------------------
    # LLM-subset class distribution
    # --------------------------------------------------------

    print(
        "\nLLM-subset class distribution:"
    )

    print(
        d.llm_test["label"]
        .value_counts()
    )

    print(
        "\nLLM-subset class percentages:"
    )

    print(
        (
            d.llm_test["label"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        )
    )

    # --------------------------------------------------------
    # Additional group information
    # --------------------------------------------------------

    print(
        "\nGroups represented in LLM subset:",
        d.llm_test["group_id"].nunique()
    )

    # Synthetic-specific diagnostic
    if name == "synthetic":

        print(
            "\nSynthetic TEST simulation runs per fault:"
        )

        print(
            (
                d.test[
                    ["group_id", "label"]
                ]
                .drop_duplicates()
                ["label"]
                .value_counts()
                .sort_index()
            )
        )

    # NASA-specific diagnostic
    if name == "nasa_fd001":

        print(
            "\nNASA windows per test engine:"
        )

        print(
            d.test["group_id"]
            .value_counts()
            .sort_index()
        )

        print(
            "\nNASA LLM-subset windows per engine:"
        )

        print(
            d.llm_test["group_id"]
            .value_counts()
            .sort_index()
        )

    print(
        "\nLeakage check: PASSED ✓"
    )

## 4. Conventional machine-learning baselines

In [ ]:
# 7. Conventional baselines on full test AND identical LLM subset
def models():
    return {
      "Logistic Regression":Pipeline([("s",StandardScaler()),("m",LogisticRegression(max_iter=3000,class_weight="balanced",random_state=42))]),
      "SVM-RBF":Pipeline([("s",StandardScaler()),("m",SVC(kernel="rbf",class_weight="balanced",probability=True,random_state=42))]),
      "Random Forest":RandomForestClassifier(n_estimators=500,class_weight="balanced",random_state=42,n_jobs=-1)
    }

BASE_MODELS={}; base_rows=[]
for name,d in PREP.items():
    for mn,m in models().items():
        m.fit(d.train[d.features],d.train.label); BASE_MODELS[(name,mn)]=m
        for scope,e in [("full_test",d.test),("llm_like_for_like",d.llm_test)]:
            p=m.predict(e[d.features])
            base_rows.append({"dataset":name,"model":mn,"scope":scope,"n":len(e),
             "accuracy":accuracy_score(e.label,p),"macro_f1":f1_score(e.label,p,average="macro"),
             "macro_precision":precision_score(e.label,p,average="macro",zero_division=0),
             "macro_recall":recall_score(e.label,p,average="macro",zero_division=0)})
BASE_DF=pd.DataFrame(base_rows); BASE_DF.to_csv(RESULTS/"baseline_metrics.csv",index=False); BASE_DF

## 5. Training-only analogical memory and independent prototype verifier

In [ ]:
# Independent deterministic verifier + training-only analogical memory

class Verifier:
    """Class-prototype verifier built exclusively from training data."""

    def __init__(self, dataset):
        self.d = dataset
        Z = dataset.scaler.transform(dataset.train[dataset.features])
        y = dataset.train.label.to_numpy()
        self.centroids = {
            label: Z[y == label].mean(axis=0)
            for label in dataset.labels
        }

    def support(self, z):
        distances = np.array([
            np.mean((z - self.centroids[label]) ** 2)
            for label in self.d.labels
        ])
        values = softmax(-distances / PROTOTYPE_TAU)
        return dict(zip(self.d.labels, values))


class Memory:
    """Cosine-similarity retrieval over labelled training instances only."""

    def __init__(self, dataset):
        self.d = dataset
        self.Z = dataset.scaler.transform(dataset.train[dataset.features])

    def retrieve(self, z, k=MEMORY_K):
        similarities = cosine_similarity(z.reshape(1, -1), self.Z)[0]
        indices = np.argsort(similarities)[::-1][:k]

        output = []
        for i in indices:
            zr = self.Z[i]
            strongest = np.argsort(np.abs(zr))[::-1][:4]
            output.append({
                "instance_id": str(self.d.train.iloc[i].instance_id),
                "label": str(self.d.train.iloc[i].label),
                "similarity": float(similarities[i]),
                "signature": [
                    (self.d.features[j], float(zr[j]))
                    for j in strongest
                ],
            })
        return output


VER = {name: Verifier(d) for name, d in PREP.items()}
MEM = {name: Memory(d) for name, d in PREP.items()}

print("Verifier/memory datasets:", sorted(VER))

## 6. Prompt, structured diagnostic traces, grounding, and raw modal aggregation

In [ ]:
# ============================================================
# Structured diagnostic-rationale prompt, trace generation, and grounding
# ============================================================

def zrow(d, row):

    z = d.scaler.transform(
        pd.DataFrame(
            [row[d.features].to_dict()]
        )
    )[0]

    return (
        z,
        {
            f: float(v)
            for f, v in zip(
                d.features,
                z
            )
        }
    )


# ============================================================
# Prompt
# ============================================================

def prompt(
    d,
    feat,
    analogs=None
):

    # --------------------------------------------------------
    # Dataset-specific task description
    # --------------------------------------------------------

    if d.name == "synthetic":

        task = (
            "Simulated manufacturing motor diagnosis. "
            "Allowed labels: "
            + ", ".join(d.labels)
            + "."
        )

    else:

        task = (
            "NASA C-MAPSS FD001 health-state diagnosis. "
            "Allowed labels: healthy, degraded. "
            "RUL is NOT provided and must not be inferred."
        )

    # --------------------------------------------------------
    # Optional analogical-memory context
    # Training-reference cases ONLY
    # --------------------------------------------------------

    memory_text = ""

    if analogs:

        lines = []

        for a in analogs:

            signature = ", ".join(
                f"{f}={v:.2f}"
                for f, v in a["signature"]
            )

            lines.append(
                f"- training reference "
                f"{a['instance_id']}: "
                f"label={a['label']}, "
                f"similarity={a['similarity']:.3f}, "
                f"{signature}"
            )

        memory_text = (
            "\nValidated TRAINING references. "
            "Use these only as weak contextual evidence; "
            "current sensor evidence has priority:\n"
            + "\n".join(lines)
        )

    # --------------------------------------------------------
    # Standardized features
    # --------------------------------------------------------

    feature_lines = "\n".join(
        f"- {k}: {v:.4f}"
        for k, v in feat.items()
    )

    # --------------------------------------------------------
    # System prompt
    # --------------------------------------------------------

    system = (
        "You are an engineering diagnostic assistant. "
        "Return a concise evidence-grounded diagnostic rationale. "
        "Do not claim access to hidden model reasoning."
    )

    # --------------------------------------------------------
    # User prompt
    # --------------------------------------------------------

    user = f"""
{task}

All feature values below are standardized using TRAINING-set
statistics only.

FEATURES:
{feature_lines}

{memory_text}

Choose exactly ONE diagnosis from:
{", ".join(d.labels)}

Return VALID JSON ONLY.

Use exactly this structure:

{{
  "diagnosis": "<one allowed label>",
  "alternative": "<one allowed label or null>",
  "evidence": [
    {{
      "feature": "<exact supplied feature name>",
      "direction": "positive|negative|near_zero",
      "importance": "high|medium|low"
    }}
  ],
  "contradictions": [],
  "rationale": "<concise engineering explanation>"
}}

STRICT RULES:

1. diagnosis must be exactly one allowed label.
2. Use ONLY feature names supplied above.
3. Do NOT invent sensor values.
4. Include between 2 and 4 evidence items only.
5. Keep the rationale below 60 words.
6. Keep contradictions empty unless there is a clear contradiction.
7. Direction definitions:
   - positive: standardized value > +{DIRECTION_EPS}
   - negative: standardized value < -{DIRECTION_EPS}
   - near_zero: between -{DIRECTION_EPS} and +{DIRECTION_EPS}
8. Return JSON only. No markdown fences.
"""

    return (
        system,
        user.strip()
    )


# ============================================================
# Evidence-grounding evaluation
# ============================================================

def actual_dir(v):

    if v > DIRECTION_EPS:
        return "positive"

    elif v < -DIRECTION_EPS:
        return "negative"

    else:
        return "near_zero"


def grounding(
    parsed,
    feat
):

    evidence = (
        parsed.get("evidence")
        or []
    )

    verifiable = 0
    supported = 0
    contradicted = 0
    invalid = 0

    for item in evidence:

        feature = item.get(
            "feature"
        )

        direction = item.get(
            "direction"
        )

        # Invalid/invented feature
        if feature not in feat:

            invalid += 1
            continue

        # Invalid direction value
        if direction not in (
            "positive",
            "negative",
            "near_zero"
        ):

            continue

        verifiable += 1

        actual = actual_dir(
            feat[feature]
        )

        if direction == actual:

            supported += 1

        else:

            contradicted += 1

    grounded_rate = (
        supported / verifiable
        if verifiable
        else np.nan
    )

    contradiction_rate = (
        contradicted / verifiable
        if verifiable
        else np.nan
    )

    unsupported_rate = (
        (contradicted + invalid)
        /
        (verifiable + invalid)
        if (verifiable + invalid)
        else np.nan
    )

    return {

        "grounded_rate":
            grounded_rate,

        "contradiction_rate":
            contradiction_rate,

        "unsupported_rate":
            unsupported_rate,

        "invalid_feature_claims":
            invalid,

        "verifiable_claims":
            verifiable
    }


# ============================================================
# Trace generation
# ============================================================

RAW = []


def traces(
    d,
    row,
    n,
    use_memory,
    experiment,
    repeat
):

    z, feat = zrow(
        d,
        row
    )

    analogs = (
        MEM[d.name].retrieve(
            z,
            MEMORY_K
        )
        if use_memory
        else None
    )

    system_prompt, user_prompt = prompt(
        d,
        feat,
        analogs
    )

    output = []

    for i in range(n):

        meta = {

            "dataset":
                d.name,

            "instance_id":
                row.instance_id,

            "experiment":
                experiment,

            "repeat":
                repeat,

            "trace_idx":
                i,

            "use_memory":
                use_memory,

            "labels":
                d.labels,

            "features":
                feat
        }

        parsed, usage = llm_call(
            system_prompt,
            user_prompt,
            meta
        )

        diagnosis = str(
            parsed.get(
                "diagnosis",
                "unknown"
            )
        ).lower().strip()

        # ----------------------------------------------------
        # Strict label validation
        # ----------------------------------------------------

        if diagnosis not in d.labels:

            diagnosis = "unknown"

        parsed["diagnosis"] = (
            diagnosis
        )

        # ----------------------------------------------------
        # Record parse validity explicitly
        # ----------------------------------------------------

        parse_success = (
            diagnosis != "unknown"
            and not parsed.get(
                "_parse_error",
                False
            )
        )

        gm = grounding(
            parsed,
            feat
        )

        trace_result = {

            **parsed,

            **gm,

            **usage,

            "parse_success":
                parse_success
        }

        output.append(
            trace_result
        )

        RAW.append({

            "dataset":
                d.name,

            "instance_id":
                row.instance_id,

            "true_label":
                row.label,

            "experiment":
                experiment,

            "repeat":
                repeat,

            "trace_idx":
                i,

            "use_memory":
                use_memory,

            "diagnosis":
                diagnosis,

            "parse_success":
                parse_success,

            "rationale":
                parsed.get(
                    "rationale",
                    ""
                ),

            "evidence_json":
                json.dumps(
                    parsed.get(
                        "evidence",
                        []
                    )
                ),

            **gm,

            **usage
        })

    return output


# ============================================================
# Corrected aggregation
# ============================================================

def aggregate(d, row, tr):
    """Aggregate sampled LLM traces without allowing verification to change the label.

    The raw prediction is always the modal valid LLM diagnosis. Agreement uses
    all requested traces in the denominator, so failed/unknown traces cannot
    inflate consistency.
    """

    total_traces = len(tr)

    if total_traces == 0:
        return {
            "prediction": "unknown",
            "modal": "unknown",
            "agreement": 0.0,
            "valid_traces": 0,
            "failed_traces": 0,
            "parse_success_rate": 0.0,
            "grounded_rate": np.nan,
            "unsupported_rate": np.nan,
            "mean_latency_s": np.nan,
            "input_tokens": 0,
            "output_tokens": 0,
            "cost_usd": np.nan,
        }

    valid_labels = [
        x["diagnosis"]
        for x in tr
        if x.get("diagnosis") in d.labels
    ]
    valid_count = len(valid_labels)
    failed_count = total_traces - valid_count
    parse_success_rate = valid_count / total_traces

    if not valid_labels:
        return {
            "prediction": "unknown",
            "modal": "unknown",
            "agreement": 0.0,
            "valid_traces": 0,
            "failed_traces": total_traces,
            "parse_success_rate": 0.0,
            "grounded_rate": np.nan,
            "unsupported_rate": np.nan,
            "mean_latency_s": float(np.nanmean([
                x.get("latency_s", np.nan) for x in tr
            ])),
            "input_tokens": sum(x.get("input_tokens", 0) for x in tr),
            "output_tokens": sum(x.get("output_tokens", 0) for x in tr),
            "cost_usd": float(np.nansum([
                x.get("cost_usd", np.nan) for x in tr
            ])),
        }

    counts = Counter(valid_labels)
    modal_label, modal_count = counts.most_common(1)[0]
    agreement = modal_count / total_traces

    grounded_values = [
        x.get("grounded_rate", np.nan)
        for x in tr
        if x.get("parse_success", False)
    ]
    unsupported_values = [
        x.get("unsupported_rate", np.nan)
        for x in tr
        if x.get("parse_success", False)
    ]

    grounded_rate = (
        float(np.nanmean(grounded_values))
        if grounded_values else np.nan
    )
    unsupported_rate = (
        float(np.nanmean(unsupported_values))
        if unsupported_values else np.nan
    )

    mean_latency = float(np.nanmean([
        x.get("latency_s", np.nan) for x in tr
    ]))
    input_tokens = sum(x.get("input_tokens", 0) for x in tr)
    output_tokens = sum(x.get("output_tokens", 0) for x in tr)
    total_cost = float(np.nansum([
        x.get("cost_usd", np.nan) for x in tr
    ]))

    return {
        "prediction": modal_label,
        "modal": modal_label,
        "agreement": float(agreement),
        "valid_traces": valid_count,
        "failed_traces": failed_count,
        "parse_success_rate": float(parse_success_rate),
        "grounded_rate": grounded_rate,
        "unsupported_rate": unsupported_rate,
        "mean_latency_s": mean_latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd": total_cost,
    }

# ============================================================
# Save representative prompt templates
# ============================================================

for name, d in PREP.items():

    _, example_feat = zrow(
        d,
        d.llm_test.iloc[0]
    )

    system_text, user_text = prompt(
        d,
        example_feat,
        None
    )

    (
        PROMPTS
        /
        f"{name}_system.txt"
    ).write_text(
        system_text
    )

    (
        PROMPTS
        /
        f"{name}_example_user_prompt.txt"
    ).write_text(
        user_text
    )


print("Prompt/trace functions loaded successfully.")

## 7. Main repeated experiment and selective governance

In [ ]:
# ============================================================
# Main repeated CoT-FD6 experiment
#
# Design:
#   - Same fixed held-out LLM subset for every repeat
#   - 100 instances per dataset
#   - 5 independent repeats
#   - 5 LLM traces per instance
#   - Raw LLM modal diagnosis retained for classification
#   - Deterministic verifier evaluated independently
#   - LLM/verifier disagreement -> human review
#   - Checkpoint after EVERY completed instance
#   - Safe resume after Colab interruption
#   - Raw traces checkpointed separately
#
# IMPORTANT:
# Running THIS cell only DEFINES the experiment.
# It does NOT start API calls.
# ============================================================


# ============================================================
# 11.1 Experiment / checkpoint paths
# ============================================================

MAIN_CSV = (
    RESULTS
    /
    "cotfd6_main_predictions.csv"
)

RAW_MAIN_CSV = (
    RESULTS
    /
    "cotfd6_main_raw_traces.csv"
)

METRICS_REPEAT_CSV = (
    RESULTS
    /
    "cotfd6_main_metrics_by_repeat.csv"
)

METRICS_MEAN_SD_CSV = (
    RESULTS
    /
    "cotfd6_main_metrics_mean_sd.csv"
)


# ============================================================
# 11.2 Experiment identity
#
# This allows us to detect accidental resumption using
# different model/configuration settings.
# ============================================================

EXPERIMENT_VERSION = (
    "major_revision_v2_governance_conflict"
)

RUN_CONFIG = {

    "experiment_version":
        EXPERIMENT_VERSION,

    "model":
        MODEL,

    "temperature":
        TEMPERATURE,

    "max_output_tokens":
        MAX_OUTPUT_TOKENS,

    "main_n":
        MAIN_N,

    "main_repeats":
        MAIN_REPEATS,

    "alpha":
        ALPHA,

    "memory_k":
        MEMORY_K,

    "split_seed":
        SPLIT_SEED
}


print(
    "Frozen main-experiment configuration:"
)

for k, v in RUN_CONFIG.items():

    print(
        f"  {k:22s}: {v}"
    )


# ============================================================
# 11.3 OPTIONAL reset
#
# LEAVE THIS FALSE for normal use.
#
# Change to True ONLY if you deliberately want to delete
# an existing main experiment and start from zero.
# ============================================================

RESET_MAIN = False


if RESET_MAIN:

    for p in [
        MAIN_CSV,
        RAW_MAIN_CSV,
        METRICS_REPEAT_CSV,
        METRICS_MEAN_SD_CSV
    ]:

        if p.exists():

            p.unlink()

            print(
                "Deleted:",
                p
            )


# ============================================================
# 11.4 Load existing checkpoint if one exists
# ============================================================

MAIN = []


def load_main_checkpoint():

    if not MAIN_CSV.exists():

        print(
            "\nNo existing main checkpoint found."
        )

        print(
            "A new experiment will start when run_main() is called."
        )

        return []

    old = pd.read_csv(
        MAIN_CSV
    )

    print(
        f"\nExisting checkpoint found: "
        f"{len(old)} completed instance-runs."
    )

    # --------------------------------------------------------
    # Protect against accidentally mixing experiments
    # --------------------------------------------------------

    required_config_cols = [
        "experiment_version",
        "model",
        "temperature",
        "max_output_tokens",
        "main_n",
        "main_repeats",
        "alpha",
        "memory_k",
        "split_seed"
    ]

    missing = [
        c
        for c in required_config_cols
        if c not in old.columns
    ]

    if missing:

        raise RuntimeError(
            "Existing main checkpoint does not contain "
            "the frozen configuration columns:\n"
            f"{missing}\n\n"
            "Do NOT continue automatically. "
            "Rename/delete the old checkpoint or set "
            "RESET_MAIN=True only if you intentionally "
            "want a fresh experiment."
        )

    # --------------------------------------------------------
    # Compare current frozen configuration with checkpoint
    # --------------------------------------------------------

    for key, expected in RUN_CONFIG.items():

        observed = (
            old[key]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        if len(observed) == 0:

            continue

        if str(expected) not in observed:

            raise RuntimeError(
                f"Configuration mismatch for '{key}'.\n"
                f"Current value : {expected}\n"
                f"Checkpoint    : {observed}\n\n"
                "Do not mix experiments."
            )

    return old.to_dict(
        orient="records"
    )


MAIN = load_main_checkpoint()


# ============================================================
# 11.5 Completed-instance index
#
# A completed unit is:
#
# dataset + instance_id + repeat
#
# If Colab disconnects, rerunning run_main() skips every
# completed unit and continues with the first unfinished one.
# ============================================================

COMPLETED = set()


for r in MAIN:

    COMPLETED.add(
        (
            str(r["dataset"]),
            str(r["instance_id"]),
            int(r["repeat"])
        )
    )


print(
    f"\nCompleted instance-runs currently available: "
    f"{len(COMPLETED)}"
)


# ============================================================
# 11.6 Independent governance / verification output
#
# CRITICAL:
#
# The verifier does NOT overwrite the raw LLM diagnosis.
#
# Raw LLM prediction:
#     modal diagnosis across generated traces
#
# Independent verifier:
#     strongest deterministic prototype-supported class
#
# Disagreement:
#     human_review
# ============================================================

def governance_decision(
    d,
    row,
    aggregated
):

    z, _ = zrow(
        d,
        row
    )

    supports = VER[
        d.name
    ].support(z)

    # --------------------------------------------------------
    # RAW LLM prediction
    # --------------------------------------------------------

    raw_llm_prediction = (
        aggregated.get(
            "modal",
            "unknown"
        )
    )

    # --------------------------------------------------------
    # Independent verifier prediction
    # --------------------------------------------------------

    verifier_prediction = max(
        supports,
        key=supports.get
    )

    verifier_support = float(
        supports[
            verifier_prediction
        ]
    )

    # --------------------------------------------------------
    # Verifier support specifically for the LLM diagnosis
    # --------------------------------------------------------

    if raw_llm_prediction in supports:

        llm_label_support = float(
            supports[
                raw_llm_prediction
            ]
        )

    else:

        llm_label_support = (
            np.nan
        )

    # --------------------------------------------------------
    # Verification disagreement
    # --------------------------------------------------------

    verification_conflict = (
        raw_llm_prediction
        != verifier_prediction
    )

    # Unknown LLM result must also be reviewed
    if raw_llm_prediction not in d.labels:

        disposition = (
            "human_review"
        )

    elif verification_conflict:

        disposition = (
            "human_review"
        )

    else:

        disposition = (
            "accepted"
        )

    # --------------------------------------------------------
    # Self-trust
    #
    # IMPORTANT:
    # T combines:
    #
    # A = cross-trace agreement
    # C = verifier support FOR THE RAW LLM LABEL
    #
    # It does NOT determine the final diagnosis.
    # --------------------------------------------------------

    agreement = float(
        aggregated.get(
            "agreement",
            np.nan
        )
    )

    if (
        np.isfinite(
            agreement
        )
        and
        np.isfinite(
            llm_label_support
        )
    ):

        self_trust = float(

            ALPHA
            *
            agreement

            +

            (1 - ALPHA)
            *
            llm_label_support
        )

    else:

        self_trust = (
            np.nan
        )

    return {

        "prediction":
            raw_llm_prediction,

        "llm_prediction":
            raw_llm_prediction,

        "verifier_prediction":
            verifier_prediction,

        "verifier_support":
            verifier_support,

        "llm_label_support":
            llm_label_support,

        # Kept for compatibility with earlier tables,
        # but scientifically interpret this as
        # LLM-label verification support.
        "confidence":
            llm_label_support,

        "self_trust":
            self_trust,

        "verification_conflict":
            bool(
                verification_conflict
            ),

        "disposition":
            disposition
    }


# ============================================================
# 11.7 Save RAW main traces safely
# ============================================================

def save_raw_main_checkpoint():

    # Current runtime traces generated by MAIN experiment only
    current_raw = pd.DataFrame(
        RAW
    )

    if len(current_raw):

        if (
            "experiment"
            in current_raw.columns
        ):

            current_raw = current_raw[
                current_raw[
                    "experiment"
                ]
                ==
                "main_full"
            ].copy()

    # --------------------------------------------------------
    # Load already saved raw traces, if any
    # --------------------------------------------------------

    if RAW_MAIN_CSV.exists():

        old_raw = pd.read_csv(
            RAW_MAIN_CSV
        )

    else:

        old_raw = pd.DataFrame()

    # --------------------------------------------------------
    # Combine old + new
    # --------------------------------------------------------

    parts = []

    if len(old_raw):

        parts.append(
            old_raw
        )

    if len(current_raw):

        parts.append(
            current_raw
        )

    if not parts:

        return

    combined = pd.concat(
        parts,
        ignore_index=True,
        sort=False
    )

    # --------------------------------------------------------
    # Remove duplicate traces
    # --------------------------------------------------------

    dedupe_cols = [

        c
        for c in [
            "dataset",
            "instance_id",
            "experiment",
            "repeat",
            "trace_idx"
        ]
        if c in combined.columns
    ]

    if dedupe_cols:

        combined = (
            combined
            .drop_duplicates(
                subset=dedupe_cols,
                keep="last"
            )
        )

    combined.to_csv(
        RAW_MAIN_CSV,
        index=False
    )


# ============================================================
# 11.8 Save instance-level checkpoint
# ============================================================

def save_main_checkpoint():

    if not MAIN:

        return

    df = pd.DataFrame(
        MAIN
    )

    df = (
        df
        .drop_duplicates(
            subset=[
                "dataset",
                "instance_id",
                "repeat"
            ],
            keep="last"
        )
        .sort_values(
            [
                "dataset",
                "repeat",
                "instance_id"
            ]
        )
        .reset_index(
            drop=True
        )
    )

    df.to_csv(
        MAIN_CSV,
        index=False
    )

    save_raw_main_checkpoint()


# ============================================================
# 11.9 Metrics for each independent repeat
# ============================================================

def repeated_metrics(df):

    out = []

    if len(df) == 0:

        return pd.DataFrame()

    for (
        dataset_name,
        rep
    ), g in df.groupby(
        [
            "dataset",
            "repeat"
        ]
    ):

        g = g.copy()

        # ----------------------------------------------------
        # Raw LLM correctness
        # ----------------------------------------------------

        correct = (
            g["true_label"]
            ==
            g["prediction"]
        )

        # ----------------------------------------------------
        # Independent verifier correctness
        # ----------------------------------------------------

        verifier_correct = (
            g["true_label"]
            ==
            g[
                "verifier_prediction"
            ]
        )

        # ----------------------------------------------------
        # Governance policy
        # ----------------------------------------------------

        conflict = (
            g[
                "verification_conflict"
            ]
            .astype(bool)
        )

        accepted = (
            g[
                "disposition"
            ]
            ==
            "accepted"
        )

        accepted_df = (
            g[
                accepted
            ]
        )

        # ----------------------------------------------------
        # Error capture
        # ----------------------------------------------------

        n_errors = int(
            (~correct)
            .sum()
        )

        captured_errors = int(
            (
                (~correct)
                &
                conflict
            )
            .sum()
        )

        error_capture_rate = (

            captured_errors
            /
            n_errors

            if n_errors > 0

            else np.nan
        )

        # ----------------------------------------------------
        # False escalation
        # ----------------------------------------------------

        n_correct = int(
            correct.sum()
        )

        false_escalations = int(
            (
                correct
                &
                conflict
            )
            .sum()
        )

        false_escalation_rate = (

            false_escalations
            /
            n_correct

            if n_correct > 0

            else np.nan
        )

        # ----------------------------------------------------
        # Selective accuracy
        # ----------------------------------------------------

        selective_accuracy = (

            accuracy_score(
                accepted_df[
                    "true_label"
                ],
                accepted_df[
                    "prediction"
                ]
            )

            if len(
                accepted_df
            ) > 0

            else np.nan
        )

        # ----------------------------------------------------
        # Build one repeat summary
        # ----------------------------------------------------

        out.append({

            "dataset":
                dataset_name,

            "repeat":
                int(rep),

            "n":
                len(g),

            # -----------------------------------------------
            # Raw diagnostic performance
            # -----------------------------------------------

            "accuracy":
                accuracy_score(
                    g["true_label"],
                    g["prediction"]
                ),

            "macro_f1":
                f1_score(
                    g["true_label"],
                    g["prediction"],
                    average="macro",
                    zero_division=0
                ),

            "macro_precision":
                precision_score(
                    g["true_label"],
                    g["prediction"],
                    average="macro",
                    zero_division=0
                ),

            "macro_recall":
                recall_score(
                    g["true_label"],
                    g["prediction"],
                    average="macro",
                    zero_division=0
                ),

            # -----------------------------------------------
            # LLM stochastic consistency
            # -----------------------------------------------

            "mean_agreement":
                g[
                    "agreement"
                ].mean(),

            "sd_instance_agreement":
                g[
                    "agreement"
                ].std(),

            # -----------------------------------------------
            # Verification
            # -----------------------------------------------

            "mean_llm_label_support":
                g[
                    "llm_label_support"
                ].mean(),

            "mean_verifier_support":
                g[
                    "verifier_support"
                ].mean(),

            "verifier_accuracy":
                verifier_correct.mean(),

            # -----------------------------------------------
            # Self-trust
            # -----------------------------------------------

            "mean_self_trust":
                g[
                    "self_trust"
                ].mean(),

            # -----------------------------------------------
            # Governance / selective prediction
            # -----------------------------------------------

            "verification_conflict_rate":
                conflict.mean(),

            "automatic_coverage":
                accepted.mean(),

            "selective_accuracy":
                selective_accuracy,

            "llm_errors":
                n_errors,

            "errors_captured":
                captured_errors,

            "error_capture_rate":
                error_capture_rate,

            "false_escalations":
                false_escalations,

            "false_escalation_rate":
                false_escalation_rate,

            # -----------------------------------------------
            # Parsing robustness
            # -----------------------------------------------

            "mean_parse_success_rate":
                g[
                    "parse_success_rate"
                ].mean(),

            "total_failed_traces":
                g[
                    "failed_traces"
                ].sum(),

            # -----------------------------------------------
            # Explanation grounding
            # -----------------------------------------------

            "mean_grounded_rate":
                g[
                    "grounded_rate"
                ].mean(),

            "mean_unsupported_rate":
                g[
                    "unsupported_rate"
                ].mean(),

            # -----------------------------------------------
            # Computational/API metrics
            # -----------------------------------------------

            "mean_latency_per_trace_s":
                g[
                    "mean_latency_s"
                ].mean(),

            "total_input_tokens":
                g[
                    "input_tokens"
                ].sum(),

            "total_output_tokens":
                g[
                    "output_tokens"
                ].sum(),

            "cost_usd":
                g[
                    "cost_usd"
                ].sum(
                    min_count=1
                )
        })

    return pd.DataFrame(
        out
    )


# ============================================================
# 11.10 Save/update summary metrics
# ============================================================

def update_main_summaries():

    if not MAIN:

        return

    main_df = pd.DataFrame(
        MAIN
    )

    main_df = (
        main_df
        .drop_duplicates(
            subset=[
                "dataset",
                "instance_id",
                "repeat"
            ],
            keep="last"
        )
    )

    metrics = repeated_metrics(
        main_df
    )

    if len(metrics) == 0:

        return

    metrics.to_csv(
        METRICS_REPEAT_CSV,
        index=False
    )

    numeric_cols = (
        metrics
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    numeric_cols = [
        c
        for c in numeric_cols
        if c != "repeat"
    ]

    mean_sd = (
        metrics
        .groupby(
            "dataset"
        )[
            numeric_cols
        ]
        .agg(
            [
                "mean",
                "std"
            ]
        )
    )

    mean_sd.to_csv(
        METRICS_MEAN_SD_CSV
    )


# ============================================================
# 11.11 MAIN runner — SAFE RESUME
# ============================================================

def run_main(d):

    global MAIN
    global COMPLETED

    print(
        "\n" + "=" * 80
    )

    print(
        f"MAIN EXPERIMENT — {d.name}"
    )

    print(
        "=" * 80
    )

    print(
        f"Instances per repeat : "
        f"{len(d.llm_test)}"
    )

    print(
        f"Independent repeats  : "
        f"{MAIN_REPEATS}"
    )

    print(
        f"Traces per instance  : "
        f"{MAIN_N}"
    )

    print(
        f"Maximum API calls    : "
        f"{len(d.llm_test) * MAIN_REPEATS * MAIN_N}"
    )

    # --------------------------------------------------------
    # Repeat loop
    # --------------------------------------------------------

    for rep in range(
        MAIN_REPEATS
    ):

        print(
            "\n" + "-" * 80
        )

        print(
            f"{d.name} — Repeat "
            f"{rep + 1}/{MAIN_REPEATS}"
        )

        print(
            "-" * 80
        )

        # ----------------------------------------------------
        # Instance loop
        # ----------------------------------------------------

        for _, row in tqdm(
            d.llm_test.iterrows(),
            total=len(
                d.llm_test
            ),
            desc=(
                f"{d.name} "
                f"main r{rep + 1}"
            )
        ):

            key = (
                str(
                    d.name
                ),
                str(
                    row.instance_id
                ),
                int(
                    rep
                )
            )

            # ------------------------------------------------
            # Resume logic
            # ------------------------------------------------

            if key in COMPLETED:

                continue

            # ------------------------------------------------
            # Generate independent LLM traces
            # ------------------------------------------------

            tr = traces(
                d,
                row,
                MAIN_N,
                True,
                "main_full",
                rep
            )

            # ------------------------------------------------
            # Base aggregation
            # ------------------------------------------------

            aggregated = aggregate(
                d,
                row,
                tr
            )

            # ------------------------------------------------
            # Independent verification / governance
            # ------------------------------------------------

            governance = governance_decision(
                d,
                row,
                aggregated
            )

            # ------------------------------------------------
            # Final instance record
            #
            # Governance adds independent verifier/support/disposition
            # fields while preserving the raw modal LLM prediction.
            # ------------------------------------------------

            record = {

                "dataset":
                    d.name,

                "instance_id":
                    row.instance_id,

                "group_id":
                    row.group_id,

                "true_label":
                    row.label,

                "repeat":
                    rep,

                **aggregated,

                **governance,

                # --------------------------------------------
                # Frozen configuration metadata
                # --------------------------------------------

                "experiment_version":
                    EXPERIMENT_VERSION,

                "model":
                    MODEL,

                "temperature":
                    TEMPERATURE,

                "max_output_tokens":
                    MAX_OUTPUT_TOKENS,

                "main_n":
                    MAIN_N,

                "main_repeats":
                    MAIN_REPEATS,

                "alpha":
                    ALPHA,

                "memory_k":
                    MEMORY_K,

                "split_seed":
                    SPLIT_SEED
            }

            MAIN.append(
                record
            )

            COMPLETED.add(
                key
            )

            # ------------------------------------------------
            # CRITICAL:
            # Save after EVERY completed diagnostic instance.
            # ------------------------------------------------

            save_main_checkpoint()

        # ----------------------------------------------------
        # End-of-repeat summary
        # ----------------------------------------------------

        update_main_summaries()

        current = pd.DataFrame(
            MAIN
        )

        current_repeat = current[
            (
                current[
                    "dataset"
                ]
                ==
                d.name
            )
            &
            (
                current[
                    "repeat"
                ]
                ==
                rep
            )
        ]

        print(
            f"\nRepeat {rep + 1} checkpoint:"
        )

        print(
            f"Completed "
            f"{len(current_repeat)}"
            f"/{len(d.llm_test)} "
            f"instances."
        )

        if (
            len(current_repeat)
            ==
            len(d.llm_test)
        ):

            quick_accuracy = (
                accuracy_score(
                    current_repeat[
                        "true_label"
                    ],
                    current_repeat[
                        "prediction"
                    ]
                )
            )

            quick_f1 = (
                f1_score(
                    current_repeat[
                        "true_label"
                    ],
                    current_repeat[
                        "prediction"
                    ],
                    average="macro",
                    zero_division=0
                )
            )

            print(
                f"Raw accuracy : "
                f"{quick_accuracy:.4f}"
            )

            print(
                f"Macro-F1     : "
                f"{quick_f1:.4f}"
            )

            print(
                f"Conflict rate: "
                f"{current_repeat['verification_conflict'].mean():.4f}"
            )

            print(
                f"Coverage     : "
                f"{(current_repeat['disposition'] == 'accepted').mean():.4f}"
            )

    # ========================================================
    # Dataset complete
    # ========================================================

    save_main_checkpoint()

    update_main_summaries()

    print(
        "\n" + "=" * 80
    )

    print(
        f"{d.name} MAIN EXPERIMENT COMPLETE"
    )

    print(
        "=" * 80
    )


# ============================================================
# 11.12 Current experiment status
# ============================================================

def main_status():

    if not MAIN:

        print(
            "No main-experiment instances "
            "have been completed yet."
        )

        return

    df = pd.DataFrame(
        MAIN
    )

    status = (
        df
        .groupby(
            [
                "dataset",
                "repeat"
            ]
        )
        .size()
        .reset_index(
            name="completed_instances"
        )
    )

    print(
        "\nCurrent MAIN experiment status:"
    )

    display(
        status
    )


main_status()


# ============================================================
# DO NOT AUTOMATICALLY START THE EXPERIMENT HERE.
#
# Run ONE of the following commands manually in a NEW cell.
# ============================================================

# run_main(PREP["nasa_fd001"])

# run_main(PREP["synthetic"])

# To run both sequentially ONLY when ready:
#
# run_main(PREP["nasa_fd001"])
# run_main(PREP["synthetic"])

### Optional main API execution

The following cell runs the full main experiment only when `COTFD6_RUN_LLM=1` was set before starting the kernel. The original reported main experiment used 100 held-out instances per dataset, 5 traces per instance, and 5 repetitions.

In [ ]:
if RUN_LLM_EXPERIMENTS:
    if "nasa_fd001" not in PREP:
        raise RuntimeError("NASA FD001 is required for the full paper reproduction.")
    run_main(PREP["nasa_fd001"])
    run_main(PREP["synthetic"])
else:
    print("Skipped main API experiment. Set COTFD6_RUN_LLM=1 to enable it.")

## 8. Trace-count sensitivity and ablation

In [ ]:
# ============================================================
# Trace-count sensitivity and ablation experiments
#
# E3:
#   N = 3, 5, 10 traces
#   Full framework held constant:
#       memory ON
#       independent verifier/governance ON
#   Only N changes
#
# E4/E5:
#   A1 = single trace, no memory, no verification
#   A2 = multi trace, no memory, no verification
#   A3 = multi trace + independent verification/governance
#   A4 = full framework: multi + verification + memory
#
# IMPORTANT:
#   - verifier NEVER overwrites raw LLM diagnosis
#   - A2 and A3 reuse the SAME LLM traces
#   - A1 reuses trace 1 of A2
#   - safe Google Drive checkpoints
#   - safe resume after Colab interruption
#
# Running this cell ONLY DEFINES the experiments.
# ZERO API calls until run_* functions are invoked.
# ============================================================


# ============================================================
# 12.1 Persistent paths
# ============================================================

SECONDARY_VERSION = (
    "major_revision_v3_secondary_governance"
)

TRACE_CSV = (
    RESULTS /
    "trace_sensitivity_predictions.csv"
)

ABL_CSV = (
    RESULTS /
    "ablation_predictions.csv"
)

SECONDARY_RAW_CSV = (
    RESULTS /
    "secondary_experiment_raw_traces.csv"
)

TRACE_SUMMARY_CSV = (
    RESULTS /
    "trace_sensitivity_metrics.csv"
)

ABL_SUMMARY_CSV = (
    RESULTS /
    "ablation_metrics.csv"
)


# ============================================================
# 12.2 Optional reset
#
# LEAVE FALSE unless we deliberately want to erase E3-E5.
# ============================================================

RESET_SECONDARY = False


if RESET_SECONDARY:

    for p in [
        TRACE_CSV,
        ABL_CSV,
        SECONDARY_RAW_CSV,
        TRACE_SUMMARY_CSV,
        ABL_SUMMARY_CSV
    ]:

        if p.exists():
            p.unlink()
            print("Deleted:", p)


# ============================================================
# 12.3 Load any existing checkpoints
# ============================================================

def load_secondary_records(path):

    if not path.exists():
        return []

    df = pd.read_csv(path)

    if len(df) == 0:
        return []

    if "secondary_version" not in df.columns:

        raise RuntimeError(
            f"{path.name} exists but is from an older "
            "experimental design.\n"
            "Do not mix old and revised E3-E5 results."
        )

    versions = (
        df["secondary_version"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if (
        len(versions)
        and
        SECONDARY_VERSION not in versions
    ):

        raise RuntimeError(
            f"Version mismatch in {path.name}: "
            f"{versions}"
        )

    return df.to_dict(
        orient="records"
    )


TRACE = load_secondary_records(
    TRACE_CSV
)

ABL = load_secondary_records(
    ABL_CSV
)


print(
    "Loaded trace-sensitivity records:",
    len(TRACE)
)

print(
    "Loaded ablation records:",
    len(ABL)
)


# ============================================================
# 12.4 Save prediction checkpoints
# ============================================================

def save_trace_checkpoint():

    if not TRACE:
        return

    df = pd.DataFrame(TRACE)

    df = (
        df
        .drop_duplicates(
            subset=[
                "dataset",
                "instance_id",
                "repeat",
                "n_traces"
            ],
            keep="last"
        )
        .sort_values(
            [
                "dataset",
                "repeat",
                "n_traces",
                "instance_id"
            ]
        )
    )

    df.to_csv(
        TRACE_CSV,
        index=False
    )


def save_ablation_checkpoint():

    if not ABL:
        return

    df = pd.DataFrame(ABL)

    df = (
        df
        .drop_duplicates(
            subset=[
                "dataset",
                "instance_id",
                "repeat",
                "variant"
            ],
            keep="last"
        )
        .sort_values(
            [
                "dataset",
                "repeat",
                "variant",
                "instance_id"
            ]
        )
    )

    df.to_csv(
        ABL_CSV,
        index=False
    )


# ============================================================
# 12.5 Persistent RAW trace checkpoint
#
# After a runtime restart RAW=[] again.
# Therefore secondary traces are also stored independently.
# ============================================================

SECONDARY_EXPERIMENT_NAMES = {

    "trace_sensitivity",

    "ablation_no_memory",

    "ablation_with_memory"
}


def save_secondary_raw_checkpoint():

    current = pd.DataFrame(
        RAW
    )

    if len(current):

        current = current[
            current["experiment"].isin(
                SECONDARY_EXPERIMENT_NAMES
            )
        ].copy()

    if SECONDARY_RAW_CSV.exists():

        old = pd.read_csv(
            SECONDARY_RAW_CSV
        )

    else:

        old = pd.DataFrame()

    parts = []

    if len(old):
        parts.append(old)

    if len(current):
        parts.append(current)

    if not parts:
        return

    combined = pd.concat(
        parts,
        ignore_index=True,
        sort=False
    )

    dedupe = [
        c
        for c in [
            "dataset",
            "instance_id",
            "experiment",
            "repeat",
            "trace_idx"
        ]
        if c in combined.columns
    ]

    if dedupe:

        combined = (
            combined
            .drop_duplicates(
                subset=dedupe,
                keep="last"
            )
        )

    combined.to_csv(
        SECONDARY_RAW_CSV,
        index=False
    )


# ============================================================
# 12.6 Raw LLM result helper
#
# aggregate(...) returns the raw modal LLM label; verification is applied separately.
# ============================================================

def raw_result(
    d,
    row,
    trace_list,
    use_governance=False
):

    base = aggregate(
        d,
        row,
        trace_list
    )

    # Raw modal LLM diagnosis
    base["prediction"] = (
        base.get(
            "modal",
            "unknown"
        )
    )

    base["llm_prediction"] = (
        base["prediction"]
    )

    if use_governance:

        gov = governance_decision(
            d,
            row,
            base
        )

        # governance_decision preserves
        # prediction = raw modal label
        result = {
            **base,
            **gov
        }

    else:

        result = {
            **base,

            "verifier_prediction":
                np.nan,

            "verifier_support":
                np.nan,

            "llm_label_support":
                np.nan,

            "verification_conflict":
                np.nan,

            "disposition":
                "not_applicable"
        }

    return result


# ============================================================
# 12.7 Completion checks for safe resume
# ============================================================

def trace_completed(
    dataset,
    instance_id,
    repeat
):

    if not TRACE:
        return False

    df = pd.DataFrame(
        TRACE
    )

    g = df[
        (df["dataset"] == dataset)
        &
        (df["instance_id"] == instance_id)
        &
        (df["repeat"] == repeat)
    ]

    completed_n = set(
        pd.to_numeric(
            g["n_traces"],
            errors="coerce"
        )
        .dropna()
        .astype(int)
        .tolist()
    )

    return (
        set(TRACE_NS)
        .issubset(
            completed_n
        )
    )


ABLATION_VARIANTS = {

    "A1_single",

    "A2_multi",

    "A3_multi_verify",

    "A4_full"
}


def ablation_completed(
    dataset,
    instance_id,
    repeat
):

    if not ABL:
        return False

    df = pd.DataFrame(
        ABL
    )

    g = df[
        (df["dataset"] == dataset)
        &
        (df["instance_id"] == instance_id)
        &
        (df["repeat"] == repeat)
    ]

    completed_variants = set(
        g["variant"]
        .astype(str)
        .tolist()
    )

    return (
        ABLATION_VARIANTS
        .issubset(
            completed_variants
        )
    )


# ============================================================
# 12.8 E3 — TRACE-NUMBER SENSITIVITY
#
# IMPORTANT:
# Generate N=10 once, then reuse the first 3/5/10.
#
# Memory remains ON because the purpose is to vary ONLY
# the number of traces relative to the full framework.
# ============================================================

def run_trace_sensitivity(d):

    sub = strat_cap(
        d.llm_test,
        TRACE_MAX,
        53
    )

    max_n = max(
        TRACE_NS
    )

    print("\n" + "=" * 80)

    print(
        f"E3 TRACE SENSITIVITY — {d.name}"
    )

    print("=" * 80)

    print(
        "Instances:",
        len(sub)
    )

    print(
        "Repeats:",
        TRACE_REPEATS
    )

    print(
        "N values:",
        TRACE_NS
    )

    print(
        "Maximum new API calls:",
        len(sub)
        * TRACE_REPEATS
        * max_n
    )

    for rep in range(
        TRACE_REPEATS
    ):

        for _, row in tqdm(
            sub.iterrows(),
            total=len(sub),
            desc=(
                f"{d.name} trace "
                f"r{rep + 1}"
            )
        ):

            if trace_completed(
                d.name,
                row.instance_id,
                rep
            ):
                continue

            # -----------------------------------------------
            # Generate maximum number ONCE
            # Full framework has training-only memory ON.
            # -----------------------------------------------

            t10 = traces(
                d,
                row,
                max_n,
                True,
                "trace_sensitivity",
                rep
            )

            for n in TRACE_NS:

                result = raw_result(
                    d,
                    row,
                    t10[:n],
                    use_governance=True
                )

                TRACE.append({

                    "dataset":
                        d.name,

                    "instance_id":
                        row.instance_id,

                    "group_id":
                        row.group_id,

                    "true_label":
                        row.label,

                    "repeat":
                        rep,

                    "n_traces":
                        n,

                    "secondary_version":
                        SECONDARY_VERSION,

                    "model":
                        MODEL,

                    "temperature":
                        TEMPERATURE,

                    **result
                })

            # Save after every diagnostic instance
            save_trace_checkpoint()

            save_secondary_raw_checkpoint()

    print(
        f"\nE3 complete/checkpointed for {d.name}."
    )


# ============================================================
# 12.9 E4/E5 — ABLATION
#
# A1:
#   one LLM trace
#
# A2:
#   five traces / modal vote
#
# A3:
#   same five traces as A2
#   + independent verification/governance
#
# A4:
#   five traces
#   + verification/governance
#   + training-only analogical memory
#
# A2 and A3 MUST share the exact same LLM traces.
# Therefore any difference in governance metrics is due to
# verification, not fresh stochastic sampling.
# ============================================================

def run_ablation(d):

    sub = strat_cap(
        d.llm_test,
        ABLATION_MAX,
        63
    )

    print("\n" + "=" * 80)

    print(
        f"E4/E5 ABLATION — {d.name}"
    )

    print("=" * 80)

    print(
        "Instances:",
        len(sub)
    )

    print(
        "Repeats:",
        ABLATION_REPEATS
    )

    print(
        "Variants:",
        sorted(
            ABLATION_VARIANTS
        )
    )

    print(
        "Maximum new API calls:",
        len(sub)
        * ABLATION_REPEATS
        * (
            MAIN_N
            +
            MAIN_N
        )
    )

    for rep in range(
        ABLATION_REPEATS
    ):

        for _, row in tqdm(
            sub.iterrows(),
            total=len(sub),
            desc=(
                f"{d.name} ablation "
                f"r{rep + 1}"
            )
        ):

            if ablation_completed(
                d.name,
                row.instance_id,
                rep
            ):
                continue

            # ===============================================
            # NO-MEMORY traces
            #
            # These exact same traces are used for A1/A2/A3.
            # ===============================================

            nm = traces(
                d,
                row,
                MAIN_N,
                False,
                "ablation_no_memory",
                rep
            )

            # -----------------------------------------------
            # A1 — SINGLE TRACE
            # -----------------------------------------------

            a1 = raw_result(
                d,
                row,
                nm[:1],
                use_governance=False
            )

            # -----------------------------------------------
            # A2 — MULTI TRACE
            # -----------------------------------------------

            a2 = raw_result(
                d,
                row,
                nm,
                use_governance=False
            )

            # -----------------------------------------------
            # A3 — SAME MULTI TRACE + VERIFICATION
            #
            # Raw diagnosis remains exactly the A2 modal label.
            # Verification contributes conflict/escalation,
            # NOT a silently corrected classification label.
            # -----------------------------------------------

            a3 = raw_result(
                d,
                row,
                nm,
                use_governance=True
            )

            # ===============================================
            # MEMORY traces for FULL configuration
            # ===============================================

            wm = traces(
                d,
                row,
                MAIN_N,
                True,
                "ablation_with_memory",
                rep
            )

            # -----------------------------------------------
            # A4 — FULL FRAMEWORK
            # -----------------------------------------------

            a4 = raw_result(
                d,
                row,
                wm,
                use_governance=True
            )

            variants = {

                "A1_single":
                    a1,

                "A2_multi":
                    a2,

                "A3_multi_verify":
                    a3,

                "A4_full":
                    a4
            }

            for variant, result in variants.items():

                ABL.append({

                    "dataset":
                        d.name,

                    "instance_id":
                        row.instance_id,

                    "group_id":
                        row.group_id,

                    "true_label":
                        row.label,

                    "repeat":
                        rep,

                    "variant":
                        variant,

                    "secondary_version":
                        SECONDARY_VERSION,

                    "model":
                        MODEL,

                    "temperature":
                        TEMPERATURE,

                    **result
                })

            save_ablation_checkpoint()

            save_secondary_raw_checkpoint()

    print(
        f"\nE4/E5 complete/checkpointed for {d.name}."
    )


# ============================================================
# 12.10 Governance-summary helper
# ============================================================

def governance_metrics(g):

    if (
        "disposition"
        not in g.columns
    ):

        return {
            "conflict_rate": np.nan,
            "coverage": np.nan,
            "selective_accuracy": np.nan,
            "error_capture_rate": np.nan,
            "false_escalation_rate": np.nan
        }

    applicable = (
        g["disposition"]
        !=
        "not_applicable"
    )

    if not applicable.any():

        return {
            "conflict_rate": np.nan,
            "coverage": np.nan,
            "selective_accuracy": np.nan,
            "error_capture_rate": np.nan,
            "false_escalation_rate": np.nan
        }

    gg = g[
        applicable
    ].copy()

    conflict = (
        gg["verification_conflict"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

    accepted = (
        gg["disposition"]
        ==
        "accepted"
    )

    correct = (
        gg["true_label"]
        ==
        gg["prediction"]
    )

    errors = (
        ~correct
    )

    n_errors = int(
        errors.sum()
    )

    n_correct = int(
        correct.sum()
    )

    captured = int(
        (
            errors
            &
            conflict
        )
        .sum()
    )

    false_esc = int(
        (
            correct
            &
            conflict
        )
        .sum()
    )

    accepted_df = gg[
        accepted
    ]

    selective_accuracy = (

        accuracy_score(
            accepted_df[
                "true_label"
            ],
            accepted_df[
                "prediction"
            ]
        )

        if len(
            accepted_df
        )

        else np.nan
    )

    return {

        "conflict_rate":
            float(
                conflict.mean()
            ),

        "coverage":
            float(
                accepted.mean()
            ),

        "selective_accuracy":
            selective_accuracy,

        "error_capture_rate":

            (
                captured
                /
                n_errors

                if n_errors

                else np.nan
            ),

        "false_escalation_rate":

            (
                false_esc
                /
                n_correct

                if n_correct

                else np.nan
            )
    }


# ============================================================
# 12.11 Build E3 summary
# ============================================================

def summarize_trace_sensitivity():

    if not TRACE:
        return pd.DataFrame()

    df = pd.DataFrame(
        TRACE
    )

    rows = []

    for (
        dataset,
        rep,
        n
    ), g in df.groupby(
        [
            "dataset",
            "repeat",
            "n_traces"
        ]
    ):

        gov = governance_metrics(
            g
        )

        rows.append({

            "dataset":
                dataset,

            "repeat":
                int(rep),

            "n_traces":
                int(n),

            "n_instances":
                len(g),

            "accuracy":
                accuracy_score(
                    g.true_label,
                    g.prediction
                ),

            "macro_f1":
                f1_score(
                    g.true_label,
                    g.prediction,
                    average="macro",
                    zero_division=0
                ),

            "mean_agreement":
                g.agreement.mean(),

            "mean_self_trust":
                g.self_trust.mean(),

            "mean_grounded_rate":
                g.grounded_rate.mean(),

            "mean_unsupported_rate":
                g.unsupported_rate.mean(),

            "parse_success_rate":
                g.parse_success_rate.mean(),

            "mean_latency_s":
                g.mean_latency_s.mean(),

            **gov
        })

    out = pd.DataFrame(
        rows
    )

    out.to_csv(
        TRACE_SUMMARY_CSV,
        index=False
    )

    return out


# ============================================================
# 12.12 Build ablation summary
# ============================================================

def summarize_ablation():

    if not ABL:
        return pd.DataFrame()

    df = pd.DataFrame(
        ABL
    )

    rows = []

    for (
        dataset,
        rep,
        variant
    ), g in df.groupby(
        [
            "dataset",
            "repeat",
            "variant"
        ]
    ):

        gov = governance_metrics(
            g
        )

        rows.append({

            "dataset":
                dataset,

            "repeat":
                int(rep),

            "variant":
                variant,

            "n_instances":
                len(g),

            "accuracy":
                accuracy_score(
                    g.true_label,
                    g.prediction
                ),

            "macro_f1":
                f1_score(
                    g.true_label,
                    g.prediction,
                    average="macro",
                    zero_division=0
                ),

            "mean_agreement":
                g.agreement.mean(),

            "mean_grounded_rate":
                g.grounded_rate.mean(),

            "mean_unsupported_rate":
                g.unsupported_rate.mean(),

            "parse_success_rate":
                g.parse_success_rate.mean(),

            "mean_latency_s":
                g.mean_latency_s.mean(),

            **gov
        })

    out = pd.DataFrame(
        rows
    )

    out.to_csv(
        ABL_SUMMARY_CSV,
        index=False
    )

    return out


# ============================================================
# 12.13 Status / summaries
# ============================================================

def secondary_status():

    print("\n" + "=" * 80)

    print(
        "SECONDARY EXPERIMENT STATUS"
    )

    print("=" * 80)

    print(
        "Trace-sensitivity records:",
        len(TRACE)
    )

    print(
        "Ablation records:",
        len(ABL)
    )

    if SECONDARY_RAW_CSV.exists():

        raw_sec = pd.read_csv(
            SECONDARY_RAW_CSV
        )

        print(
            "Saved secondary raw traces:",
            len(raw_sec)
        )

        if "cost_usd" in raw_sec.columns:

            print(
                "Actual API cost recorded so far: $",
                round(
                    raw_sec[
                        "cost_usd"
                    ].sum(),
                    4
                )
            )


secondary_status()


print(
    "\nCell 12 final E3-E5 definitions loaded."
)

print(
    "NO API calls have been made by running this definition cell."
)

**Trace-sensitivity configuration used for the reported run:** training-only analogical memory remained **ON** while only the number of sampled traces was varied. This preserves the actual executed configuration.

In [ ]:
if RUN_LLM_EXPERIMENTS:
    if "nasa_fd001" not in PREP:
        raise RuntimeError("NASA FD001 is required for the full paper reproduction.")
    run_trace_sensitivity(PREP["nasa_fd001"])
    run_trace_sensitivity(PREP["synthetic"])
    run_ablation(PREP["nasa_fd001"])
    run_ablation(PREP["synthetic"])
else:
    print("Skipped trace-sensitivity and ablation API experiments.")

## 9. Retrieval-only baseline and integrity analyses

In [ ]:
# ============================================================
# MEMORY INTEGRITY + MEMORY-ONLY BASELINE AUDIT
# ZERO API CALLS
# ============================================================

from collections import Counter

MEM_AUDIT = []

for name, d in PREP.items():

    # EXACT SAME subset used by the ablation experiment
    sub = strat_cap(
        d.llm_test,
        ABLATION_MAX,
        63
    )

    train_ids = set(
        d.train["instance_id"]
        .astype(str)
    )

    train_group_by_id = dict(
        zip(
            d.train["instance_id"].astype(str),
            d.train["group_id"].astype(str)
        )
    )

    print("\n" + "=" * 80)
    print("MEMORY AUDIT —", name)
    print("=" * 80)

    # Global group-leakage check
    train_groups = set(
        d.train["group_id"]
        .astype(str)
    )

    eval_groups = set(
        sub["group_id"]
        .astype(str)
    )

    group_overlap = (
        train_groups
        &
        eval_groups
    )

    print(
        "Train/evaluation group overlap:",
        len(group_overlap)
    )

    for _, row in sub.iterrows():

        z, _ = zrow(
            d,
            row
        )

        refs = MEM[name].retrieve(
            z,
            MEMORY_K
        )

        ref_ids = [
            str(a["instance_id"])
            for a in refs
        ]

        ref_labels = [
            str(a["label"])
            for a in refs
        ]

        similarities = [
            float(a["similarity"])
            for a in refs
        ]

        # --------------------------------------------
        # Simple majority-label memory-only predictor
        # --------------------------------------------

        majority_label = (
            Counter(ref_labels)
            .most_common(1)[0][0]
        )

        # --------------------------------------------
        # Similarity-weighted memory-only predictor
        # --------------------------------------------

        weighted_scores = {}

        for lab, sim in zip(
            ref_labels,
            similarities
        ):

            weighted_scores[lab] = (
                weighted_scores.get(
                    lab,
                    0.0
                )
                +
                sim
            )

        weighted_label = max(
            weighted_scores,
            key=weighted_scores.get
        )

        # --------------------------------------------
        # Leakage / integrity checks
        # --------------------------------------------

        self_reference = (
            str(row.instance_id)
            in ref_ids
        )

        all_refs_training = all(
            rid in train_ids
            for rid in ref_ids
        )

        same_group_reference = any(
            train_group_by_id.get(
                rid,
                "__missing__"
            )
            ==
            str(row.group_id)

            for rid in ref_ids
        )

        # How unanimous are the retrieved labels?
        memory_label_agreement = (
            Counter(ref_labels)
            .most_common(1)[0][1]
            /
            len(ref_labels)
        )

        MEM_AUDIT.append({

            "dataset":
                name,

            "instance_id":
                str(row.instance_id),

            "group_id":
                str(row.group_id),

            "true_label":
                str(row.label),

            "memory_majority_prediction":
                majority_label,

            "memory_weighted_prediction":
                weighted_label,

            "memory_label_agreement":
                memory_label_agreement,

            "self_reference":
                self_reference,

            "all_refs_training":
                all_refs_training,

            "same_group_reference":
                same_group_reference,

            "reference_ids":
                json.dumps(ref_ids),

            "reference_labels":
                json.dumps(ref_labels),

            "reference_similarities":
                json.dumps(similarities)
        })


MEM_AUDIT_DF = pd.DataFrame(
    MEM_AUDIT
)


# ============================================================
# Memory-only performance summary
# ============================================================

summary_rows = []

for dataset, g in MEM_AUDIT_DF.groupby(
    "dataset"
):

    majority_acc = accuracy_score(
        g["true_label"],
        g["memory_majority_prediction"]
    )

    majority_f1 = f1_score(
        g["true_label"],
        g["memory_majority_prediction"],
        average="macro",
        zero_division=0
    )

    weighted_acc = accuracy_score(
        g["true_label"],
        g["memory_weighted_prediction"]
    )

    weighted_f1 = f1_score(
        g["true_label"],
        g["memory_weighted_prediction"],
        average="macro",
        zero_division=0
    )

    summary_rows.append({

        "dataset":
            dataset,

        "n_instances":
            len(g),

        "self_references":
            int(
                g["self_reference"]
                .sum()
            ),

        "nontraining_references":
            int(
                (~g["all_refs_training"])
                .sum()
            ),

        "same_group_references":
            int(
                g["same_group_reference"]
                .sum()
            ),

        "mean_memory_label_agreement":
            g[
                "memory_label_agreement"
            ].mean(),

        "memory_majority_accuracy":
            majority_acc,

        "memory_majority_macro_f1":
            majority_f1,

        "memory_weighted_accuracy":
            weighted_acc,

        "memory_weighted_macro_f1":
            weighted_f1
    })


MEM_AUDIT_SUMMARY = pd.DataFrame(
    summary_rows
)


# ============================================================
# Compare A4 LLM prediction with memory-only prediction
# ============================================================

ABL_NOW = pd.DataFrame(
    ABL
)

A4 = ABL_NOW[
    ABL_NOW["variant"]
    ==
    "A4_full"
].copy()

A4["instance_id"] = (
    A4["instance_id"]
    .astype(str)
)

COMPARE = A4.merge(

    MEM_AUDIT_DF[
        [
            "dataset",
            "instance_id",
            "memory_majority_prediction",
            "memory_weighted_prediction"
        ]
    ],

    on=[
        "dataset",
        "instance_id"
    ],

    how="left"
)

A4_MEMORY_COMPARE = (

    COMPARE
    .assign(

        matches_majority=lambda x:
            (
                x["prediction"]
                ==
                x[
                    "memory_majority_prediction"
                ]
            ),

        matches_weighted=lambda x:
            (
                x["prediction"]
                ==
                x[
                    "memory_weighted_prediction"
                ]
            )
    )

    .groupby(
        "dataset"
    )

    .agg(

        n_a4_predictions=(
            "prediction",
            "size"
        ),

        a4_matches_memory_majority=(
            "matches_majority",
            "mean"
        ),

        a4_matches_memory_weighted=(
            "matches_weighted",
            "mean"
        )
    )

    .reset_index()
)


# ============================================================
# Display + save
# ============================================================

print("\nMEMORY-ONLY PERFORMANCE / INTEGRITY")
display(
    MEM_AUDIT_SUMMARY
)

print("\nA4 LLM vs MEMORY-ONLY PREDICTIONS")
display(
    A4_MEMORY_COMPARE
)

MEM_AUDIT_DF.to_csv(
    RESULTS /
    "memory_integrity_instance_audit.csv",
    index=False
)

MEM_AUDIT_SUMMARY.to_csv(
    RESULTS /
    "memory_integrity_summary.csv",
    index=False
)

A4_MEMORY_COMPARE.to_csv(
    RESULTS /
    "a4_vs_memory_only.csv",
    index=False
)

print(
    "\nMemory audit saved successfully."
)

print(
    "ZERO API calls were made."
)

In [ ]:
# ============================================================
# A4 vs MEMORY-ONLY DISAGREEMENT AUDIT
# ZERO API CALLS
# ============================================================

A4_DETAIL = A4.merge(

    MEM_AUDIT_DF[
        [
            "dataset",
            "instance_id",
            "true_label",
            "memory_majority_prediction"
        ]
    ],

    on=[
        "dataset",
        "instance_id"
    ],

    how="left",

    suffixes=(
        "_a4",
        "_memory"
    )
)


A4_DETAIL["a4_correct"] = (
    A4_DETAIL["prediction"]
    ==
    A4_DETAIL["true_label_a4"]
)

A4_DETAIL["memory_correct"] = (
    A4_DETAIL["memory_majority_prediction"]
    ==
    A4_DETAIL["true_label_a4"]
)

A4_DETAIL["a4_memory_agree"] = (
    A4_DETAIL["prediction"]
    ==
    A4_DETAIL["memory_majority_prediction"]
)


rows = []

for dataset, g in A4_DETAIL.groupby("dataset"):

    both_correct = (
        g["a4_correct"]
        &
        g["memory_correct"]
    ).sum()

    memory_only_correct = (
        (~g["a4_correct"])
        &
        g["memory_correct"]
    ).sum()

    a4_only_correct = (
        g["a4_correct"]
        &
        (~g["memory_correct"])
    ).sum()

    both_wrong = (
        (~g["a4_correct"])
        &
        (~g["memory_correct"])
    ).sum()

    disagree = (
        ~g["a4_memory_agree"]
    ).sum()

    rows.append({

        "dataset":
            dataset,

        "n_predictions":
            len(g),

        "a4_memory_disagreements":
            int(disagree),

        "both_correct":
            int(both_correct),

        "memory_correct_a4_wrong":
            int(memory_only_correct),

        "a4_correct_memory_wrong":
            int(a4_only_correct),

        "both_wrong":
            int(both_wrong)
    })


A4_DISAGREEMENT_SUMMARY = pd.DataFrame(
    rows
)

display(
    A4_DISAGREEMENT_SUMMARY
)


# Also show per-repeat NASA performance
PER_REPEAT_MEMORY_COMPARE = []

for (dataset, rep), g in A4_DETAIL.groupby(
    ["dataset", "repeat"]
):

    PER_REPEAT_MEMORY_COMPARE.append({

        "dataset":
            dataset,

        "repeat":
            rep,

        "a4_accuracy":
            g["a4_correct"].mean(),

        "memory_accuracy":
            g["memory_correct"].mean(),

        "agreement_between_a4_and_memory":
            g["a4_memory_agree"].mean()
    })


PER_REPEAT_MEMORY_COMPARE = pd.DataFrame(
    PER_REPEAT_MEMORY_COMPARE
)

display(
    PER_REPEAT_MEMORY_COMPARE
)


A4_DISAGREEMENT_SUMMARY.to_csv(
    RESULTS /
    "a4_memory_disagreement_summary.csv",
    index=False
)

PER_REPEAT_MEMORY_COMPARE.to_csv(
    RESULTS /
    "a4_memory_per_repeat_comparison.csv",
    index=False
)

print(
    "\nA4/memory disagreement audit saved."
)

print(
    "ZERO API calls were made."
)

In [ ]:
# ============================================================
# FULL 100-INSTANCE RETRIEVAL-ONLY BASELINE
# + LIKE-FOR-LIKE COMPARISON WITH MAIN LLM EXPERIMENT
#
# ZERO API CALLS
# ============================================================

from collections import Counter

MEM_FULL = []

for name, d in PREP.items():

    e = d.llm_test.copy()

    print("\n" + "=" * 80)
    print("FULL MEMORY-ONLY BASELINE —", name)
    print("=" * 80)

    print("Evaluation instances:", len(e))

    train_groups = set(
        d.train["group_id"].astype(str)
    )

    test_groups = set(
        e["group_id"].astype(str)
    )

    print(
        "Train/test group overlap:",
        len(train_groups & test_groups)
    )

    train_ids = set(
        d.train["instance_id"].astype(str)
    )

    train_group_by_id = dict(
        zip(
            d.train["instance_id"].astype(str),
            d.train["group_id"].astype(str)
        )
    )

    for _, row in e.iterrows():

        z, _ = zrow(
            d,
            row
        )

        refs = MEM[name].retrieve(
            z,
            MEMORY_K
        )

        ref_ids = [
            str(x["instance_id"])
            for x in refs
        ]

        ref_labels = [
            str(x["label"])
            for x in refs
        ]

        similarities = [
            float(x["similarity"])
            for x in refs
        ]

        # --------------------------------------------
        # Majority-label retrieval baseline
        # --------------------------------------------

        majority_prediction = (
            Counter(ref_labels)
            .most_common(1)[0][0]
        )

        # --------------------------------------------
        # Similarity-weighted retrieval baseline
        # --------------------------------------------

        scores = {}

        for lab, sim in zip(
            ref_labels,
            similarities
        ):

            scores[lab] = (
                scores.get(lab, 0.0)
                +
                sim
            )

        weighted_prediction = max(
            scores,
            key=scores.get
        )

        MEM_FULL.append({

            "dataset":
                name,

            "instance_id":
                str(row.instance_id),

            "group_id":
                str(row.group_id),

            "true_label":
                str(row.label),

            "memory_majority_prediction":
                majority_prediction,

            "memory_weighted_prediction":
                weighted_prediction,

            "memory_label_agreement":
                (
                    Counter(ref_labels)
                    .most_common(1)[0][1]
                    /
                    len(ref_labels)
                ),

            "self_reference":
                str(row.instance_id)
                in ref_ids,

            "all_refs_training":
                all(
                    rid in train_ids
                    for rid in ref_ids
                ),

            "same_group_reference":
                any(
                    train_group_by_id.get(
                        rid,
                        "__missing__"
                    )
                    ==
                    str(row.group_id)

                    for rid in ref_ids
                ),

            "reference_ids":
                json.dumps(ref_ids),

            "reference_labels":
                json.dumps(ref_labels),

            "reference_similarities":
                json.dumps(similarities)
        })


MEM_FULL_DF = pd.DataFrame(
    MEM_FULL
)


# ============================================================
# 1. Retrieval-only baseline metrics
# ============================================================

MEM_FULL_SUMMARY = []

for dataset, g in MEM_FULL_DF.groupby(
    "dataset"
):

    MEM_FULL_SUMMARY.append({

        "dataset":
            dataset,

        "n_instances":
            len(g),

        "majority_accuracy":
            accuracy_score(
                g.true_label,
                g.memory_majority_prediction
            ),

        "majority_macro_f1":
            f1_score(
                g.true_label,
                g.memory_majority_prediction,
                average="macro",
                zero_division=0
            ),

        "weighted_accuracy":
            accuracy_score(
                g.true_label,
                g.memory_weighted_prediction
            ),

        "weighted_macro_f1":
            f1_score(
                g.true_label,
                g.memory_weighted_prediction,
                average="macro",
                zero_division=0
            ),

        "mean_memory_label_agreement":
            g.memory_label_agreement.mean(),

        "self_references":
            int(
                g.self_reference.sum()
            ),

        "nontraining_references":
            int(
                (~g.all_refs_training).sum()
            ),

        "same_group_references":
            int(
                g.same_group_reference.sum()
            )
    })


MEM_FULL_SUMMARY = pd.DataFrame(
    MEM_FULL_SUMMARY
)


# ============================================================
# 2. Compare retrieval-only against each MAIN LLM repeat
# ============================================================

MAIN_CURRENT = pd.read_csv(
    MAIN_CSV
)

MAIN_CURRENT["instance_id"] = (
    MAIN_CURRENT["instance_id"]
    .astype(str)
)

MEM_FULL_DF["instance_id"] = (
    MEM_FULL_DF["instance_id"]
    .astype(str)
)


LLM_MEMORY_COMPARE = []

for (
    dataset,
    rep
), g in MAIN_CURRENT.groupby(
    [
        "dataset",
        "repeat"
    ]
):

    mem = MEM_FULL_DF[
        MEM_FULL_DF.dataset
        ==
        dataset
    ][
        [
            "instance_id",
            "true_label",
            "memory_majority_prediction"
        ]
    ]

    c = g.merge(
        mem,
        on="instance_id",
        how="inner",
        suffixes=(
            "_llm",
            "_memory"
        )
    )

    llm_correct = (
        c.prediction
        ==
        c.true_label_llm
    )

    mem_correct = (
        c.memory_majority_prediction
        ==
        c.true_label_llm
    )

    agree = (
        c.prediction
        ==
        c.memory_majority_prediction
    )

    disagreements = (
        ~agree
    )

    LLM_MEMORY_COMPARE.append({

        "dataset":
            dataset,

        "repeat":
            int(rep),

        "n_instances":
            len(c),

        "llm_accuracy":
            llm_correct.mean(),

        "memory_accuracy":
            mem_correct.mean(),

        "llm_memory_agreement":
            agree.mean(),

        "n_disagreements":
            int(
                disagreements.sum()
            ),

        "memory_correct_llm_wrong":
            int(
                (
                    mem_correct
                    &
                    (~llm_correct)
                ).sum()
            ),

        "llm_correct_memory_wrong":
            int(
                (
                    llm_correct
                    &
                    (~mem_correct)
                ).sum()
            ),

        "both_wrong":
            int(
                (
                    (~llm_correct)
                    &
                    (~mem_correct)
                ).sum()
            )
    })


LLM_MEMORY_COMPARE = pd.DataFrame(
    LLM_MEMORY_COMPARE
)


# ============================================================
# 3. Conventional baselines on SAME 100 instances
# ============================================================

LIKE_FOR_LIKE = []

for name, d in PREP.items():

    e = d.llm_test.copy()

    for model_name in [
        "Logistic Regression",
        "SVM-RBF",
        "Random Forest"
    ]:

        pred = BASE_MODELS[
            (
                name,
                model_name
            )
        ].predict(
            e[d.features]
        )

        LIKE_FOR_LIKE.append({

            "dataset":
                name,

            "model":
                model_name,

            "n_instances":
                len(e),

            "accuracy":
                accuracy_score(
                    e.label,
                    pred
                ),

            "macro_f1":
                f1_score(
                    e.label,
                    pred,
                    average="macro",
                    zero_division=0
                )
        })

    mg = MEM_FULL_DF[
        MEM_FULL_DF.dataset
        ==
        name
    ]

    LIKE_FOR_LIKE.append({

        "dataset":
            name,

        "model":
            "Retrieval-only memory",

        "n_instances":
            len(mg),

        "accuracy":
            accuracy_score(
                mg.true_label,
                mg.memory_majority_prediction
            ),

        "macro_f1":
            f1_score(
                mg.true_label,
                mg.memory_majority_prediction,
                average="macro",
                zero_division=0
            )
    })


LIKE_FOR_LIKE = pd.DataFrame(
    LIKE_FOR_LIKE
)


# ============================================================
# Display
# ============================================================

print(
    "\nFULL MEMORY-ONLY SUMMARY"
)

display(
    MEM_FULL_SUMMARY
)

print(
    "\nMAIN LLM vs MEMORY-ONLY BY REPEAT"
)

display(
    LLM_MEMORY_COMPARE
)

print(
    "\nLIKE-FOR-LIKE NON-STOCHASTIC BASELINES"
)

display(
    LIKE_FOR_LIKE
)


# ============================================================
# Save
# ============================================================

MEM_FULL_DF.to_csv(
    RESULTS /
    "full100_memory_only_predictions.csv",
    index=False
)

MEM_FULL_SUMMARY.to_csv(
    RESULTS /
    "full100_memory_only_summary.csv",
    index=False
)

LLM_MEMORY_COMPARE.to_csv(
    RESULTS /
    "full100_llm_vs_memory_by_repeat.csv",
    index=False
)

LIKE_FOR_LIKE.to_csv(
    RESULTS /
    "full100_like_for_like_baselines.csv",
    index=False
)

print(
    "\nFull 100-instance memory audit saved."
)

print(
    "ZERO API calls were made."
)

## 10. RF-SHAP and explanation-grounding analyses

In [ ]:
# ============================================================
# 13. E6 RANDOM FOREST + SHAP
#     E7 EXPLANATION GROUNDING + LLM/SHAP FEATURE OVERLAP
#
# ZERO OPENAI API CALLS
#
# Outputs:
#   1. RF SHAP top features for every held-out LLM instance
#   2. Global RF SHAP feature importance
#   3. Grounding summary from PERSISTENT raw traces
#   4. LLM-evidence vs SHAP-top-feature overlap
#
# IMPORTANT:
#   - SHAP is an XAI baseline, not "ground truth explanation"
#   - overlap measures feature-level correspondence only
#   - it does NOT establish CoT faithfulness
# ============================================================

import shap
import ast


# ============================================================
# 13.1 Load persistent raw traces
# ============================================================

raw_parts = []


# Main experiment traces: 5000 expected
if RAW_MAIN_CSV.exists():

    RAW_MAIN_PERSISTENT = pd.read_csv(
        RAW_MAIN_CSV
    )

    raw_parts.append(
        RAW_MAIN_PERSISTENT
    )

    print(
        "Loaded persistent MAIN raw traces:",
        len(RAW_MAIN_PERSISTENT)
    )

else:

    RAW_MAIN_PERSISTENT = pd.DataFrame()

    print(
        "WARNING: MAIN raw-trace checkpoint not found."
    )


# Secondary E3-E5 traces: 3600 expected
if SECONDARY_RAW_CSV.exists():

    RAW_SECONDARY_PERSISTENT = pd.read_csv(
        SECONDARY_RAW_CSV
    )

    raw_parts.append(
        RAW_SECONDARY_PERSISTENT
    )

    print(
        "Loaded persistent SECONDARY raw traces:",
        len(RAW_SECONDARY_PERSISTENT)
    )

else:

    RAW_SECONDARY_PERSISTENT = pd.DataFrame()

    print(
        "WARNING: secondary raw-trace checkpoint not found."
    )


if not raw_parts:

    raise RuntimeError(
        "No persistent raw traces were found."
    )


RAW_ALL = pd.concat(
    raw_parts,
    ignore_index=True,
    sort=False
)


# Remove accidental duplicate traces
dedupe_cols = [
    c
    for c in [
        "dataset",
        "instance_id",
        "experiment",
        "repeat",
        "trace_idx"
    ]
    if c in RAW_ALL.columns
]

if dedupe_cols:

    RAW_ALL = (
        RAW_ALL
        .drop_duplicates(
            subset=dedupe_cols,
            keep="last"
        )
        .reset_index(drop=True)
    )


print(
    "Total unique persistent raw traces:",
    len(RAW_ALL)
)


# Save unified trace file
RAW_ALL.to_csv(
    RESULTS /
    "reasoning_traces_and_grounding.csv",
    index=False
)


# ============================================================
# 13.2 Grounding summary
# ============================================================

GROUNDING_SUMMARY = (

    RAW_ALL

    .groupby(
        [
            "dataset",
            "experiment",
            "use_memory"
        ],
        dropna=False
    )

    .agg(

        n_traces=(
            "diagnosis",
            "size"
        ),

        parse_success_rate=(
            "parse_success",
            "mean"
        ),

        grounded_mean=(
            "grounded_rate",
            "mean"
        ),

        grounded_sd=(
            "grounded_rate",
            "std"
        ),

        unsupported_mean=(
            "unsupported_rate",
            "mean"
        ),

        unsupported_sd=(
            "unsupported_rate",
            "std"
        ),

        contradiction_mean=(
            "contradiction_rate",
            "mean"
        ),

        invalid_feature_claims=(
            "invalid_feature_claims",
            "sum"
        ),

        verifiable_claims=(
            "verifiable_claims",
            "sum"
        ),

        latency_mean_s=(
            "latency_s",
            "mean"
        )
    )

    .reset_index()
)


GROUNDING_SUMMARY.to_csv(
    RESULTS /
    "explanation_grounding_summary.csv",
    index=False
)


print("\n" + "=" * 80)
print("EXPLANATION GROUNDING SUMMARY")
print("=" * 80)

display(
    GROUNDING_SUMMARY
)


# ============================================================
# 13.3 RF + SHAP per held-out instance
# ============================================================

SHAP_ROWS = []
GLOBAL_SHAP_ROWS = []


for name, d in PREP.items():

    print("\n" + "=" * 80)

    print(
        "RF + SHAP —",
        name
    )

    print("=" * 80)

    rf = BASE_MODELS[
        (
            name,
            "Random Forest"
        )
    ]

    e = (
        d.llm_test
        .copy()
        .reset_index(drop=True)
    )

    X = e[
        d.features
    ]

    rf_pred = rf.predict(
        X
    )

    classes = list(
        rf.classes_
    )

    explainer = shap.TreeExplainer(
        rf
    )

    shap_values = explainer.shap_values(
        X
    )


    # --------------------------------------------------------
    # Convert SHAP output to:
    # one vector for the class predicted by RF for each row
    # --------------------------------------------------------

    row_vectors = []

    for i in range(
        len(X)
    ):

        class_idx = classes.index(
            rf_pred[i]
        )

        if isinstance(
            shap_values,
            list
        ):

            vals = np.asarray(
                shap_values[
                    class_idx
                ][i]
            )

        else:

            a = np.asarray(
                shap_values
            )

            if a.ndim == 3:

                # Common modern SHAP format:
                # samples x features x classes
                vals = a[
                    i,
                    :,
                    class_idx
                ]

            elif a.ndim == 2:

                vals = a[i]

            else:

                raise RuntimeError(
                    "Unexpected SHAP output shape: "
                    f"{a.shape}"
                )

        vals = np.asarray(
            vals,
            dtype=float
        )

        row_vectors.append(
            vals
        )

        top_idx = (
            np.argsort(
                np.abs(vals)
            )[::-1][:5]
        )

        SHAP_ROWS.append({

            "dataset":
                name,

            "instance_id":
                str(
                    e.iloc[i]
                    .instance_id
                ),

            "true_label":
                str(
                    e.iloc[i]
                    .label
                ),

            "rf_prediction":
                str(
                    rf_pred[i]
                ),

            "rf_correct":
                bool(
                    rf_pred[i]
                    ==
                    e.iloc[i]
                    .label
                ),

            "shap_top1_feature":
                d.features[
                    top_idx[0]
                ],

            "shap_top3_features":
                json.dumps(
                    [
                        d.features[j]
                        for j in top_idx[:3]
                    ]
                ),

            "shap_top5_features":
                json.dumps(
                    [
                        d.features[j]
                        for j in top_idx
                    ]
                ),

            "shap_top5_abs_values":
                json.dumps(
                    [
                        float(
                            abs(
                                vals[j]
                            )
                        )
                        for j in top_idx
                    ]
                )
        })


    # --------------------------------------------------------
    # Global mean absolute SHAP
    # --------------------------------------------------------

    M = np.vstack(
        row_vectors
    )

    mean_abs = np.mean(
        np.abs(M),
        axis=0
    )

    order = np.argsort(
        mean_abs
    )[::-1]

    for rank, j in enumerate(
        order,
        start=1
    ):

        GLOBAL_SHAP_ROWS.append({

            "dataset":
                name,

            "rank":
                rank,

            "feature":
                d.features[j],

            "mean_abs_shap":
                float(
                    mean_abs[j]
                )
        })


SHAP_DF = pd.DataFrame(
    SHAP_ROWS
)

GLOBAL_SHAP_DF = pd.DataFrame(
    GLOBAL_SHAP_ROWS
)


SHAP_DF.to_csv(
    RESULTS /
    "rf_shap_top_features.csv",
    index=False
)

GLOBAL_SHAP_DF.to_csv(
    RESULTS /
    "rf_shap_global_importance.csv",
    index=False
)


print("\nRF SHAP — GLOBAL FEATURE IMPORTANCE")

display(
    GLOBAL_SHAP_DF
    .groupby(
        "dataset",
        group_keys=False
    )
    .head(10)
)


# ============================================================
# 13.4 Helper to parse LLM evidence feature names
# ============================================================

def parse_evidence_features(x):

    if pd.isna(x):
        return []

    try:

        parsed = json.loads(
            x
        )

    except Exception:

        try:

            parsed = ast.literal_eval(
                str(x)
            )

        except Exception:

            return []

    if not isinstance(
        parsed,
        list
    ):
        return []

    out = []

    for item in parsed:

        if not isinstance(
            item,
            dict
        ):
            continue

        feature = item.get(
            "feature"
        )

        if feature is not None:

            out.append(
                str(feature)
            )

    return out


# ============================================================
# 13.5 Compare MAIN LLM evidence with RF SHAP
#
# We use main_full only:
# same fixed 100 held-out instances per dataset.
#
# IMPORTANT:
# This is feature correspondence, NOT proof of
# explanation faithfulness.
# ============================================================

MAIN_RAW_FOR_XAI = (
    RAW_MAIN_PERSISTENT[
        RAW_MAIN_PERSISTENT[
            "experiment"
        ]
        ==
        "main_full"
    ]
    .copy()
)


MAIN_RAW_FOR_XAI[
    "instance_id"
] = (
    MAIN_RAW_FOR_XAI[
        "instance_id"
    ]
    .astype(str)
)

SHAP_DF[
    "instance_id"
] = (
    SHAP_DF[
        "instance_id"
    ]
    .astype(str)
)


XAI_COMPARE = MAIN_RAW_FOR_XAI.merge(

    SHAP_DF[
        [
            "dataset",
            "instance_id",
            "rf_prediction",
            "shap_top1_feature",
            "shap_top3_features",
            "shap_top5_features"
        ]
    ],

    on=[
        "dataset",
        "instance_id"
    ],

    how="left"
)


OVERLAP_ROWS = []


for _, r in XAI_COMPARE.iterrows():

    llm_features = (
        parse_evidence_features(
            r.get(
                "evidence_json",
                "[]"
            )
        )
    )

    llm_unique = list(
        dict.fromkeys(
            llm_features
        )
    )

    try:

        shap3 = set(
            json.loads(
                r[
                    "shap_top3_features"
                ]
            )
        )

    except Exception:

        shap3 = set()

    try:

        shap5 = set(
            json.loads(
                r[
                    "shap_top5_features"
                ]
            )
        )

    except Exception:

        shap5 = set()


    llm_set = set(
        llm_unique
    )

    n_llm = len(
        llm_set
    )

    overlap3 = (
        llm_set
        &
        shap3
    )

    overlap5 = (
        llm_set
        &
        shap5
    )


    OVERLAP_ROWS.append({

        "dataset":
            r["dataset"],

        "instance_id":
            r["instance_id"],

        "repeat":
            r["repeat"],

        "trace_idx":
            r["trace_idx"],

        "diagnosis":
            r["diagnosis"],

        "rf_prediction":
            r["rf_prediction"],

        "n_llm_evidence_features":
            n_llm,

        "llm_evidence_features":
            json.dumps(
                llm_unique
            ),

        "shap_top1_feature":
            r[
                "shap_top1_feature"
            ],

        "n_overlap_top3":
            len(
                overlap3
            ),

        "n_overlap_top5":
            len(
                overlap5
            ),

        "any_overlap_top3":
            bool(
                overlap3
            ),

        "any_overlap_top5":
            bool(
                overlap5
            ),

        "llm_feature_fraction_in_shap_top3":
            (
                len(overlap3)
                /
                n_llm
                if n_llm
                else np.nan
            ),

        "llm_feature_fraction_in_shap_top5":
            (
                len(overlap5)
                /
                n_llm
                if n_llm
                else np.nan
            ),

        "shap_top1_used_by_llm":
            (
                r[
                    "shap_top1_feature"
                ]
                in
                llm_set
                if n_llm
                else False
            ),

        "grounded_rate":
            r[
                "grounded_rate"
            ],

        "unsupported_rate":
            r[
                "unsupported_rate"
            ]
    })


LLM_SHAP_OVERLAP_DF = pd.DataFrame(
    OVERLAP_ROWS
)


LLM_SHAP_SUMMARY = (

    LLM_SHAP_OVERLAP_DF

    .groupby(
        "dataset"
    )

    .agg(

        n_traces=(
            "trace_idx",
            "size"
        ),

        any_overlap_top3_rate=(
            "any_overlap_top3",
            "mean"
        ),

        any_overlap_top5_rate=(
            "any_overlap_top5",
            "mean"
        ),

        mean_llm_fraction_in_shap_top3=(
            "llm_feature_fraction_in_shap_top3",
            "mean"
        ),

        mean_llm_fraction_in_shap_top5=(
            "llm_feature_fraction_in_shap_top5",
            "mean"
        ),

        shap_top1_used_rate=(
            "shap_top1_used_by_llm",
            "mean"
        ),

        grounded_rate=(
            "grounded_rate",
            "mean"
        ),

        unsupported_rate=(
            "unsupported_rate",
            "mean"
        )
    )

    .reset_index()
)


LLM_SHAP_OVERLAP_DF.to_csv(
    RESULTS /
    "llm_shap_feature_overlap.csv",
    index=False
)

LLM_SHAP_SUMMARY.to_csv(
    RESULTS /
    "llm_shap_feature_overlap_summary.csv",
    index=False
)


print("\n" + "=" * 80)
print("LLM EVIDENCE vs RANDOM-FOREST SHAP")
print("=" * 80)

display(
    LLM_SHAP_SUMMARY
)


# ============================================================
# 13.6 Integrity information
# ============================================================

print("\n" + "=" * 80)
print("CELL 13 INTEGRITY")
print("=" * 80)

print(
    "Persistent main traces:",
    len(
        RAW_MAIN_PERSISTENT
    )
)

print(
    "Persistent secondary traces:",
    len(
        RAW_SECONDARY_PERSISTENT
    )
)

print(
    "Unified unique traces:",
    len(
        RAW_ALL
    )
)

print(
    "SHAP instances:",
    len(
        SHAP_DF
    )
)

print(
    "LLM/SHAP main trace comparisons:",
    len(
        LLM_SHAP_OVERLAP_DF
    )
)

print(
    "\nCell 13 complete."
)

print(
    "ZERO OpenAI API calls were made."
)

## 11. NASA confusion counts, cost sensitivity, and accepted-subset audit

In [ ]:
# ============================================================
# 14. E8 NASA RAW CONFUSION COUNTS
#     + FN/FP COST SENSITIVITY
#     + GOVERNANCE ACCEPTED-SUBSET AUDIT
#
# ZERO OPENAI API CALLS
#
# Positive class = "degraded"
#
# IMPORTANT:
# - Raw LLM predictions are NEVER replaced by verifier labels.
# - Cost analysis is performed on the full 100-instance subset.
# - Accepted-subset governance results are reported separately
#   because selective coverage < 100%.
# ============================================================

from sklearn.metrics import confusion_matrix


# ============================================================
# 14.1 Reload authoritative persistent main predictions
# ============================================================

MAIN14_PATH = (
    RESULTS /
    "cotfd6_main_predictions.csv"
)

if not MAIN14_PATH.exists():

    raise RuntimeError(
        "Persistent main prediction file not found: "
        f"{MAIN14_PATH}"
    )


MAIN14 = pd.read_csv(
    MAIN14_PATH
)

NASA_MAIN14 = (
    MAIN14[
        MAIN14["dataset"]
        ==
        "nasa_fd001"
    ]
    .copy()
)


print(
    "Persistent NASA main rows:",
    len(NASA_MAIN14)
)

print(
    "Unique NASA instances:",
    NASA_MAIN14[
        "instance_id"
    ].nunique()
)

print(
    "NASA repeats:",
    sorted(
        NASA_MAIN14[
            "repeat"
        ].unique()
    )
)


# Expected:
# 500 rows = 100 instances x 5 repeats
if len(NASA_MAIN14) != 500:

    raise RuntimeError(
        "Expected 500 NASA main rows, "
        f"found {len(NASA_MAIN14)}."
    )


# ============================================================
# 14.2 Reload retrieval-only predictions
# ============================================================

MEM14_PATH = (
    RESULTS /
    "full100_memory_only_predictions.csv"
)

if not MEM14_PATH.exists():

    raise RuntimeError(
        "Full memory-only prediction file not found: "
        f"{MEM14_PATH}"
    )


MEM14 = pd.read_csv(
    MEM14_PATH
)

NASA_MEM14 = (
    MEM14[
        MEM14["dataset"]
        ==
        "nasa_fd001"
    ]
    .copy()
)


print(
    "NASA retrieval-only instances:",
    len(NASA_MEM14)
)


# ============================================================
# 14.3 Confusion helper
#
# labels:
#   healthy  = negative
#   degraded = positive
#
# Therefore:
# TN = healthy -> healthy
# FP = healthy -> degraded
# FN = degraded -> healthy
# TP = degraded -> degraded
# ============================================================

def safe_div(
    numerator,
    denominator
):

    return (
        numerator / denominator
        if denominator
        else np.nan
    )


def confusion_row(
    y_true,
    y_pred,
    model,
    repeat=None,
    category="full_test"
):

    cm = confusion_matrix(

        y_true,
        y_pred,

        labels=[
            "healthy",
            "degraded"
        ]
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    n = (
        tn
        +
        fp
        +
        fn
        +
        tp
    )

    return {

        "model":
            model,

        "repeat":
            repeat,

        "category":
            category,

        "n_instances":
            int(n),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp),

        "accuracy":
            safe_div(
                tn + tp,
                n
            ),

        "healthy_recall_specificity":
            safe_div(
                tn,
                tn + fp
            ),

        "degraded_recall_sensitivity":
            safe_div(
                tp,
                tp + fn
            ),

        "precision_degraded":
            safe_div(
                tp,
                tp + fp
            )
    }


# ============================================================
# 14.4 Full 100-instance raw confusion results
# ============================================================

CONF_ROWS = []

d = PREP[
    "nasa_fd001"
]

e = (
    d.llm_test
    .copy()
)


# ------------------------------------------------------------
# Conventional classifiers
# ------------------------------------------------------------

for model_name in [

    "Logistic Regression",
    "SVM-RBF",
    "Random Forest"

]:

    pred = BASE_MODELS[
        (
            "nasa_fd001",
            model_name
        )
    ].predict(
        e[
            d.features
        ]
    )

    CONF_ROWS.append(

        confusion_row(

            e["label"],

            pred,

            model_name,

            repeat=None,

            category="full_test"
        )
    )


# ------------------------------------------------------------
# Retrieval-only baseline
# ------------------------------------------------------------

CONF_ROWS.append(

    confusion_row(

        NASA_MEM14[
            "true_label"
        ],

        NASA_MEM14[
            "memory_majority_prediction"
        ],

        "Retrieval-only memory",

        repeat=None,

        category="full_test"
    )
)


# ------------------------------------------------------------
# Independent deterministic verifier
#
# It is identical across stochastic LLM repeats because
# it depends only on the fixed instance features.
# Therefore use ONE repeat only.
# ------------------------------------------------------------

first_repeat = int(
    NASA_MAIN14[
        "repeat"
    ].min()
)

VERIFIER_ONCE = (
    NASA_MAIN14[
        NASA_MAIN14[
            "repeat"
        ]
        ==
        first_repeat
    ]
    .copy()
)


if (
    "verifier_prediction"
    in
    VERIFIER_ONCE.columns
):

    CONF_ROWS.append(

        confusion_row(

            VERIFIER_ONCE[
                "true_label"
            ],

            VERIFIER_ONCE[
                "verifier_prediction"
            ],

            "Prototype verifier (component)",

            repeat=None,

            category="full_test"
        )
    )


# ------------------------------------------------------------
# RAW LLM: five stochastic repeats
# ------------------------------------------------------------

for rep, g in NASA_MAIN14.groupby(
    "repeat"
):

    CONF_ROWS.append(

        confusion_row(

            g[
                "true_label"
            ],

            g[
                "prediction"
            ],

            "LLM raw diagnosis",

            repeat=int(rep),

            category="full_test"
        )
    )


CONF_DF = pd.DataFrame(
    CONF_ROWS
)


# ============================================================
# 14.5 Governance accepted-subset confusion
#
# This is deliberately separate from the 100-instance
# cost comparison because unaccepted cases are escalated
# rather than automatically diagnosed.
# ============================================================

GOV_ROWS = []


for rep, g in NASA_MAIN14.groupby(
    "repeat"
):

    accepted = (
        g[
            g[
                "disposition"
            ]
            ==
            "accepted"
        ]
        .copy()
    )

    if len(
        accepted
    ) == 0:

        continue

    row = confusion_row(

        accepted[
            "true_label"
        ],

        accepted[
            "prediction"
        ],

        "Governance-accepted LLM",

        repeat=int(rep),

        category="accepted_subset"
    )

    row[
        "coverage"
    ] = (
        len(accepted)
        /
        len(g)
    )

    row[
        "escalated_instances"
    ] = (
        len(g)
        -
        len(accepted)
    )

    GOV_ROWS.append(
        row
    )


GOV_CONF_DF = pd.DataFrame(
    GOV_ROWS
)


# ============================================================
# 14.6 FN/FP cost sensitivity
#
# No monetary costs are asserted.
# FP cost is normalized to 1.
# FN/FP ratios tested:
#
# 1:1
# 2:1
# 5:1
# 10:1
#
# Relative loss:
#
# L = C_FN * FN + C_FP * FP
#
# All rows here use the complete fixed 100-instance subset.
# ============================================================

COST_ROWS = []


for _, r in CONF_DF.iterrows():

    for fn_cost in [
        1,
        2,
        5,
        10
    ]:

        fp_cost = 1

        relative_loss = (
            fn_cost
            *
            r[
                "FN"
            ]
            +
            fp_cost
            *
            r[
                "FP"
            ]
        )

        COST_ROWS.append({

            "model":
                r[
                    "model"
                ],

            "repeat":
                r[
                    "repeat"
                ],

            "n_instances":
                r[
                    "n_instances"
                ],

            "FN":
                r[
                    "FN"
                ],

            "FP":
                r[
                    "FP"
                ],

            "c_fn":
                fn_cost,

            "c_fp":
                fp_cost,

            "relative_loss":
                relative_loss,

            "relative_loss_per_instance":
                (
                    relative_loss
                    /
                    r[
                        "n_instances"
                    ]
                )
        })


COST_DF = pd.DataFrame(
    COST_ROWS
)


# ============================================================
# 14.7 Mean ± SD across stochastic LLM repeats
# ============================================================

LLM_CONF_SUMMARY = (

    CONF_DF[
        CONF_DF[
            "model"
        ]
        ==
        "LLM raw diagnosis"
    ]

    .agg({

        "TN":
            [
                "mean",
                "std"
            ],

        "FP":
            [
                "mean",
                "std"
            ],

        "FN":
            [
                "mean",
                "std"
            ],

        "TP":
            [
                "mean",
                "std"
            ],

        "accuracy":
            [
                "mean",
                "std"
            ],

        "healthy_recall_specificity":
            [
                "mean",
                "std"
            ],

        "degraded_recall_sensitivity":
            [
                "mean",
                "std"
            ]
    })
)


COST_SUMMARY = (

    COST_DF

    .groupby(
        [
            "model",
            "c_fn",
            "c_fp"
        ],
        dropna=False
    )

    .agg(

        n_runs=(
            "relative_loss",
            "size"
        ),

        relative_loss_mean=(
            "relative_loss",
            "mean"
        ),

        relative_loss_sd=(
            "relative_loss",
            "std"
        ),

        relative_loss_per_instance_mean=(
            "relative_loss_per_instance",
            "mean"
        ),

        relative_loss_per_instance_sd=(
            "relative_loss_per_instance",
            "std"
        )
    )

    .reset_index()
)


GOV_SUMMARY = (

    GOV_CONF_DF

    .agg({

        "coverage":
            [
                "mean",
                "std"
            ],

        "accuracy":
            [
                "mean",
                "std"
            ],

        "healthy_recall_specificity":
            [
                "mean",
                "std"
            ],

        "degraded_recall_sensitivity":
            [
                "mean",
                "std"
            ],

        "escalated_instances":
            [
                "mean",
                "std"
            ]
    })

    if len(
        GOV_CONF_DF
    )

    else pd.DataFrame()
)


# ============================================================
# 14.8 Save
# ============================================================

CONF_DF.to_csv(
    RESULTS /
    "nasa_raw_confusion_counts.csv",
    index=False
)

COST_DF.to_csv(
    RESULTS /
    "nasa_cost_sensitivity.csv",
    index=False
)

COST_SUMMARY.to_csv(
    RESULTS /
    "nasa_cost_sensitivity_summary.csv",
    index=False
)

GOV_CONF_DF.to_csv(
    RESULTS /
    "nasa_governance_accepted_confusion.csv",
    index=False
)


# ============================================================
# 14.9 Display
# ============================================================

print(
    "\n"
    +
    "=" * 80
)

print(
    "NASA RAW CONFUSION COUNTS — SAME 100 INSTANCES"
)

print(
    "=" * 80
)

display(
    CONF_DF
)


print(
    "\nLLM RAW CONFUSION — MEAN / SD ACROSS 5 REPEATS"
)

display(
    LLM_CONF_SUMMARY
)


print(
    "\n"
    +
    "=" * 80
)

print(
    "NASA FN/FP COST SENSITIVITY"
)

print(
    "=" * 80
)

display(
    COST_SUMMARY
)


print(
    "\n"
    +
    "=" * 80
)

print(
    "NASA GOVERNANCE — ACCEPTED SUBSET ONLY"
)

print(
    "=" * 80
)

display(
    GOV_CONF_DF
)

print(
    "\nGOVERNANCE MEAN / SD"
)

display(
    GOV_SUMMARY
)


print(
    "\nCell 14 complete."
)

print(
    "ZERO OpenAI API calls were made."
)

## 12. Temporal diagnostic stability and API-cost summary

In [ ]:
# ============================================================
# Persistent API/cost summary + temporal diagnostic stability definitions
#
# IMPORTANT:
# Running this definition/preview cell makes ZERO API calls.
#
# Temporal design:
# - NASA FD001 only
# - 3 deterministic test engines
# - 5 DISTINCT sequential windows per engine
# - centered around healthy -> degraded transition
# - 5 independent LLM traces per window
# - raw LLM modal diagnosis preserved
# - verifier disagreement => human review
#
# Maximum temporal API calls:
# 3 x 5 x 5 = 75
# ============================================================


# ============================================================
# 15.1 Paths
# ============================================================

TEMP_VERSION = (
    "major_revision_v2_temporal_transition"
)

TEMP_CSV = (
    RESULTS /
    "nasa_temporal_diagnostic_stability.csv"
)

TEMP_RAW_CSV = (
    RESULTS /
    "nasa_temporal_raw_traces.csv"
)

API_SUMMARY_CSV = (
    RESULTS /
    "api_cost_latency_summary.csv"
)


# ============================================================
# 15.2 Persistent API / token / latency / cost summary
#
# IMPORTANT:
# Do not rely on runtime API_LOGS after a Colab restart.
# Use persistent raw-trace checkpoints instead.
# ============================================================

def persistent_api_summary(
    include_temporal=True
):

    parts = []

    main_path = (
        RESULTS /
        "cotfd6_main_raw_traces.csv"
    )

    secondary_path = (
        RESULTS /
        "secondary_experiment_raw_traces.csv"
    )

    if main_path.exists():

        x = pd.read_csv(
            main_path
        )

        x["source_block"] = (
            "main"
        )

        parts.append(
            x
        )


    if secondary_path.exists():

        x = pd.read_csv(
            secondary_path
        )

        x["source_block"] = (
            "secondary"
        )

        parts.append(
            x
        )


    if (
        include_temporal
        and
        TEMP_RAW_CSV.exists()
    ):

        x = pd.read_csv(
            TEMP_RAW_CSV
        )

        x["source_block"] = (
            "temporal"
        )

        parts.append(
            x
        )


    if not parts:

        raise RuntimeError(
            "No persistent raw-trace files found."
        )


    api = pd.concat(
        parts,
        ignore_index=True,
        sort=False
    )


    dedupe_cols = [
        c
        for c in [
            "dataset",
            "instance_id",
            "experiment",
            "repeat",
            "trace_idx"
        ]
        if c in api.columns
    ]


    if dedupe_cols:

        api = (
            api
            .drop_duplicates(
                subset=dedupe_cols,
                keep="last"
            )
            .reset_index(drop=True)
        )


    summary = (

        api

        .groupby(
            [
                "dataset",
                "experiment",
                "use_memory"
            ],
            dropna=False
        )

        .agg(

            calls=(
                "success",
                "size"
            ),

            success_rate=(
                "success",
                "mean"
            ),

            latency_mean_s=(
                "latency_s",
                "mean"
            ),

            latency_sd_s=(
                "latency_s",
                "std"
            ),

            input_tokens=(
                "input_tokens",
                "sum"
            ),

            output_tokens=(
                "output_tokens",
                "sum"
            ),

            estimated_cost_usd=(
                "cost_usd",
                "sum"
            )
        )

        .reset_index()
    )


    summary.to_csv(
        API_SUMMARY_CSV,
        index=False
    )


    print(
        "\nTotal persistent API traces:",
        len(api)
    )

    print(
        "Total input tokens:",
        int(
            api[
                "input_tokens"
            ].sum()
        )
    )

    print(
        "Total output tokens:",
        int(
            api[
                "output_tokens"
            ].sum()
        )
    )

    print(
        "Total estimated API cost: $",
        round(
            api[
                "cost_usd"
            ].sum(),
            4
        )
    )


    return (
        api,
        summary
    )


# ============================================================
# 15.3 Deterministically select genuine temporal windows
#
# Selection rule:
# - test engines only
# - must contain healthy AND degraded windows
# - locate FIRST degraded window
# - require 2 windows before + 2 after
# - choose first 3 qualifying engines by sorted engine ID
#
# This yields five DISTINCT sequential windows:
#
#   t-2, t-1, transition, t+1, t+2
#
# No RUL value is passed to the LLM.
# ============================================================

def select_temporal_windows(
    d,
    max_units=3
):

    selected = []

    candidate_units = sorted(
        d.test[
            "group_id"
        ].unique()
    )


    for unit in candidate_units:

        u = (
            d.test[
                d.test[
                    "group_id"
                ]
                ==
                unit
            ]
            .sort_values(
                "window_end"
            )
            .reset_index(drop=True)
        )


        labels = set(
            u[
                "label"
            ].astype(str)
        )


        if not {
            "healthy",
            "degraded"
        }.issubset(
            labels
        ):

            continue


        degraded_positions = np.where(
            u[
                "label"
            ]
            .astype(str)
            .to_numpy()
            ==
            "degraded"
        )[0]


        if len(
            degraded_positions
        ) == 0:

            continue


        transition_idx = int(
            degraded_positions[0]
        )


        # Need two windows before and after
        if (
            transition_idx < 2
            or
            transition_idx + 2 >= len(u)
        ):

            continue


        w = (
            u.iloc[
                transition_idx - 2:
                transition_idx + 3
            ]
            .copy()
            .reset_index(drop=True)
        )


        # Safety checks:
        # genuinely distinct windows only
        if (
            w[
                "instance_id"
            ].nunique()
            !=
            5
        ):

            continue


        if (
            w[
                "window_end"
            ].nunique()
            !=
            5
        ):

            continue


        w[
            "unit"
        ] = unit

        w[
            "window_order"
        ] = np.arange(
            1,
            6
        )


        selected.append(
            w
        )


        if (
            len(
                selected
            )
            ==
            max_units
        ):

            break


    if (
        len(
            selected
        )
        <
        max_units
    ):

        raise RuntimeError(
            "Could not find 3 test engines "
            "with five valid windows around "
            "the healthy/degraded transition."
        )


    out = pd.concat(
        selected,
        ignore_index=True
    )


    return out


# ============================================================
# 15.4 Preview — ZERO API CALLS
# ============================================================

NASA_TEMP_SELECTION = (
    select_temporal_windows(
        PREP[
            "nasa_fd001"
        ],
        max_units=3
    )
)


TEMP_PREVIEW_COLS = [

    "unit",
    "window_order",
    "instance_id",
    "window_start",
    "window_end",
    "rul_end",
    "label"

]


print(
    "\n"
    +
    "=" * 80
)

print(
    "E10 TEMPORAL WINDOW PREVIEW — ZERO API CALLS"
)

print(
    "=" * 80
)


display(
    NASA_TEMP_SELECTION[
        TEMP_PREVIEW_COLS
    ]
)


print(
    "\nSelected engines:",
    list(
        NASA_TEMP_SELECTION[
            "unit"
        ].drop_duplicates()
    )
)

print(
    "Selected windows:",
    len(
        NASA_TEMP_SELECTION
    )
)

print(
    "Unique instance IDs:",
    NASA_TEMP_SELECTION[
        "instance_id"
    ].nunique()
)

print(
    "Unique window ends:",
    NASA_TEMP_SELECTION[
        "window_end"
    ].nunique()
)


print(
    "\nMaximum NEW temporal API calls:",
    len(
        NASA_TEMP_SELECTION
    )
    *
    MAIN_N
)


print(
    "\nCell 15 definitions and preview loaded."
)

print(
    "NO API calls have been made."
)

In [ ]:
# ============================================================
# Temporal diagnostic-stability execution
#
# 3 engines x 5 windows x 5 traces = 75 API calls maximum
#
# IMPORTANT:
# - raw LLM modal diagnosis is preserved
# - aggregate(..., verify=False)
# - independent governance applied afterward
# - temporal raw traces saved before marking instance complete
# ============================================================


# ============================================================
# Helpers
# ============================================================

def _bool_value(v):

    if isinstance(v, str):
        return (
            v.strip()
            .lower()
            in (
                "true",
                "1",
                "yes"
            )
        )

    return bool(v)


def load_temporal_predictions():

    if not TEMP_CSV.exists():

        return pd.DataFrame()

    df = pd.read_csv(
        TEMP_CSV
    )

    if len(df):

        if (
            "temporal_version"
            not in df.columns
        ):

            raise RuntimeError(
                "Existing temporal checkpoint has no "
                "temporal_version. Do NOT mix old results."
            )

        versions = set(
            df[
                "temporal_version"
            ].astype(str)
        )

        if versions != {
            TEMP_VERSION
        }:

            raise RuntimeError(
                "Existing temporal checkpoint belongs "
                f"to another version: {versions}"
            )

    return df


def load_temporal_raw():

    if not TEMP_RAW_CSV.exists():

        return pd.DataFrame()

    df = pd.read_csv(
        TEMP_RAW_CSV
    )

    if len(df):

        if (
            "temporal_version"
            not in df.columns
        ):

            raise RuntimeError(
                "Existing temporal raw checkpoint has "
                "no temporal_version."
            )

        versions = set(
            df[
                "temporal_version"
            ].astype(str)
        )

        if versions != {
            TEMP_VERSION
        }:

            raise RuntimeError(
                "Existing temporal raw checkpoint belongs "
                f"to another version: {versions}"
            )

    return df


def save_temporal_raw(
    new_raw
):

    if new_raw is None:
        return

    if not len(new_raw):
        return

    old = load_temporal_raw()

    combined = pd.concat(
        [
            old,
            new_raw
        ],
        ignore_index=True,
        sort=False
    )

    dedupe = [
        c
        for c in [
            "dataset",
            "instance_id",
            "experiment",
            "repeat",
            "trace_idx",
            "temporal_version"
        ]
        if c in combined.columns
    ]

    if dedupe:

        combined = (
            combined
            .drop_duplicates(
                subset=dedupe,
                keep="last"
            )
            .reset_index(drop=True)
        )

    combined.to_csv(
        TEMP_RAW_CSV,
        index=False
    )


def save_temporal_prediction(
    record
):

    old = load_temporal_predictions()

    new = pd.DataFrame(
        [
            record
        ]
    )

    combined = pd.concat(
        [
            old,
            new
        ],
        ignore_index=True,
        sort=False
    )

    combined = (
        combined
        .drop_duplicates(
            subset=[
                "dataset",
                "instance_id",
                "temporal_version"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )

    combined.to_csv(
        TEMP_CSV,
        index=False
    )


# ============================================================
# Reconstruct aggregation input from already-persisted raw
# traces if a disconnect happened AFTER trace saving but
# BEFORE the diagnostic-window record was saved.
#
# This can prevent unnecessary duplicate API calls.
# ============================================================

def traces_from_temporal_raw(
    raw_df
):

    tr = []

    raw_df = (
        raw_df
        .sort_values(
            "trace_idx"
        )
    )

    for _, r in raw_df.iterrows():

        tr.append({

            "diagnosis":
                str(
                    r[
                        "diagnosis"
                    ]
                ),

            "parse_success":
                _bool_value(
                    r[
                        "parse_success"
                    ]
                ),

            "grounded_rate":
                float(
                    r[
                        "grounded_rate"
                    ]
                )
                if pd.notna(
                    r[
                        "grounded_rate"
                    ]
                )
                else np.nan,

            "unsupported_rate":
                float(
                    r[
                        "unsupported_rate"
                    ]
                )
                if pd.notna(
                    r[
                        "unsupported_rate"
                    ]
                )
                else np.nan,

            "latency_s":
                float(
                    r[
                        "latency_s"
                    ]
                )
                if pd.notna(
                    r[
                        "latency_s"
                    ]
                )
                else np.nan,

            "input_tokens":
                int(
                    r[
                        "input_tokens"
                    ]
                )
                if pd.notna(
                    r[
                        "input_tokens"
                    ]
                )
                else 0,

            "output_tokens":
                int(
                    r[
                        "output_tokens"
                    ]
                )
                if pd.notna(
                    r[
                        "output_tokens"
                    ]
                )
                else 0,

            "cost_usd":
                float(
                    r[
                        "cost_usd"
                    ]
                )
                if pd.notna(
                    r[
                        "cost_usd"
                    ]
                )
                else np.nan
        })

    return tr


# ============================================================
# Temporal runner
# ============================================================

def run_temporal_final():

    d = PREP[
        "nasa_fd001"
    ]

    completed_df = (
        load_temporal_predictions()
    )

    completed_ids = set()

    if len(
        completed_df
    ):

        completed_ids = set(
            completed_df[
                "instance_id"
            ].astype(str)
        )


    print(
        "\n"
        +
        "=" * 80
    )

    print(
        "E10 GENUINE TEMPORAL DIAGNOSTIC STABILITY"
    )

    print(
        "=" * 80
    )

    print(
        "Temporal version:",
        TEMP_VERSION
    )

    print(
        "Selected windows:",
        len(
            NASA_TEMP_SELECTION
        )
    )

    print(
        "Already completed:",
        len(
            completed_ids
        )
    )

    print(
        "Maximum remaining API calls:",
        (
            len(
                NASA_TEMP_SELECTION
            )
            -
            len(
                completed_ids
            )
        )
        *
        MAIN_N
    )


    for _, row in tqdm(

        NASA_TEMP_SELECTION.iterrows(),

        total=len(
            NASA_TEMP_SELECTION
        ),

        desc="NASA temporal stability"
    ):

        instance_id = str(
            row[
                "instance_id"
            ]
        )


        # ----------------------------------------------------
        # Already completed
        # ----------------------------------------------------

        if (
            instance_id
            in
            completed_ids
        ):

            continue


        # ----------------------------------------------------
        # Check whether complete raw traces were saved during
        # an earlier interrupted run.
        # ----------------------------------------------------

        raw_saved = (
            load_temporal_raw()
        )

        existing_instance_raw = (
            pd.DataFrame()
        )


        if len(
            raw_saved
        ):

            existing_instance_raw = (

                raw_saved[
                    (
                        raw_saved[
                            "instance_id"
                        ]
                        .astype(str)
                        ==
                        instance_id
                    )
                    &
                    (
                        raw_saved[
                            "experiment"
                        ]
                        .astype(str)
                        ==
                        "temporal_stability"
                    )
                ]

                .copy()
            )


        complete_raw_checkpoint = (

            len(
                existing_instance_raw
            )
            ==
            MAIN_N

            and

            existing_instance_raw[
                "trace_idx"
            ].nunique()
            ==
            MAIN_N
        )


        # ----------------------------------------------------
        # Either reconstruct from saved traces
        # or make the five new API calls.
        # ----------------------------------------------------

        if complete_raw_checkpoint:

            print(
                "\nRecovering saved traces for",
                instance_id,
                "— no new API calls."
            )

            tr = traces_from_temporal_raw(
                existing_instance_raw
            )


        else:

            raw_before = len(
                RAW
            )

            tr = traces(

                d,
                row,

                MAIN_N,

                True,

                "temporal_stability",

                0
            )


            # ------------------------------------------------
            # Capture ONLY the traces generated for this
            # temporal diagnostic window.
            # ------------------------------------------------

            generated_raw = pd.DataFrame(
                RAW[
                    raw_before:
                ]
            )


            if len(
                generated_raw
            ) != MAIN_N:

                raise RuntimeError(
                    "Expected "
                    f"{MAIN_N} raw temporal traces for "
                    f"{instance_id}, found "
                    f"{len(generated_raw)}."
                )


            generated_raw[
                "temporal_version"
            ] = TEMP_VERSION

            generated_raw[
                "unit"
            ] = row[
                "unit"
            ]

            generated_raw[
                "window_order"
            ] = row[
                "window_order"
            ]

            generated_raw[
                "window_start"
            ] = row[
                "window_start"
            ]

            generated_raw[
                "window_end"
            ] = row[
                "window_end"
            ]

            generated_raw[
                "rul_end"
            ] = row[
                "rul_end"
            ]


            # Save expensive evidence FIRST.
            save_temporal_raw(
                generated_raw
            )


        # ----------------------------------------------------
        # RAW LLM aggregation only.
        #
        # CRITICAL:
        # verify=False prevents the old verifier-assisted
        # aggregate() path from modifying the raw label.
        # ----------------------------------------------------

        base = aggregate(

            d,
            row,
            tr,

            False
        )


        # ----------------------------------------------------
        # Independent governance afterward.
        # ----------------------------------------------------

        governance = (
            governance_decision(

                d,
                row,
                base
            )
        )


        record = {

            "dataset":
                "nasa_fd001",

            "instance_id":
                instance_id,

            "unit":
                row[
                    "unit"
                ],

            "group_id":
                row[
                    "group_id"
                ],

            "window_order":
                int(
                    row[
                        "window_order"
                    ]
                ),

            "window_start":
                int(
                    row[
                        "window_start"
                    ]
                ),

            "window_end":
                int(
                    row[
                        "window_end"
                    ]
                ),

            # Evaluation metadata only.
            # Never supplied to the LLM.
            "rul_end":
                int(
                    row[
                        "rul_end"
                    ]
                ),

            "true_label":
                str(
                    row[
                        "label"
                    ]
                ),

            **base,

            **governance,

            "temporal_version":
                TEMP_VERSION,

            "model":
                MODEL,

            "temperature":
                TEMPERATURE,

            "n_traces":
                MAIN_N,

            "memory_k":
                MEMORY_K
        }


        # ----------------------------------------------------
        # Mark window complete only AFTER raw trace saving.
        # ----------------------------------------------------

        save_temporal_prediction(
            record
        )

        completed_ids.add(
            instance_id
        )


    final = load_temporal_predictions()


    print(
        "\nTemporal execution complete."
    )

    print(
        "Completed windows:",
        len(
            final
        ),
        "/",
        len(
            NASA_TEMP_SELECTION
        )
    )

    print(
        "Saved temporal raw traces:",
        len(
            load_temporal_raw()
        )
    )


    return final


# The function is intentionally not called automatically.
# Use the guarded execution cell below when an API rerun is intended.

### Optional temporal API execution

The reported temporal experiment used 3 held-out engines × 5 distinct windows × 5 traces = 75 API calls.

In [ ]:
if RUN_LLM_EXPERIMENTS:
    TEMP_FINAL = run_temporal_final()
else:
    print("Skipped temporal API experiment.")

### Temporal and final API summaries

In [ ]:
# ============================================================
# 15.6 TEMPORAL STABILITY SUMMARY + FINAL API SUMMARY
# ZERO NEW API CALLS
# ============================================================

TEMP_RESULT = pd.read_csv(
    TEMP_CSV
)

TEMP_RESULT = (
    TEMP_RESULT
    .sort_values(
        [
            "unit",
            "window_order"
        ]
    )
    .reset_index(drop=True)
)


TEMP_RESULT[
    "correct"
] = (
    TEMP_RESULT[
        "prediction"
    ]
    ==
    TEMP_RESULT[
        "true_label"
    ]
)


TEMP_RESULT[
    "accepted"
] = (
    TEMP_RESULT[
        "disposition"
    ]
    ==
    "accepted"
)


# ============================================================
# Per-engine temporal metrics
# ============================================================

UNIT_ROWS = []
same_state_adjacent = []


for unit, g in TEMP_RESULT.groupby(
    "unit"
):

    g = (
        g
        .sort_values(
            "window_order"
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Temporal stability where the benchmark state itself
    # does NOT change.
    #
    # We intentionally exclude the healthy->degraded boundary.
    # --------------------------------------------------------

    local_same_state = []

    for i in range(
        1,
        len(g)
    ):

        if (
            g.loc[
                i,
                "true_label"
            ]
            ==
            g.loc[
                i - 1,
                "true_label"
            ]
        ):

            stable = (
                g.loc[
                    i,
                    "prediction"
                ]
                ==
                g.loc[
                    i - 1,
                    "prediction"
                ]
            )

            local_same_state.append(
                stable
            )

            same_state_adjacent.append(
                stable
            )


    true_degraded = (
        g[
            g[
                "true_label"
            ]
            ==
            "degraded"
        ][
            "window_order"
        ]
    )


    predicted_degraded = (
        g[
            g[
                "prediction"
            ]
            ==
            "degraded"
        ][
            "window_order"
        ]
    )


    true_transition_order = (
        int(
            true_degraded.min()
        )
        if len(
            true_degraded
        )
        else np.nan
    )


    first_predicted_degraded_order = (
        int(
            predicted_degraded.min()
        )
        if len(
            predicted_degraded
        )
        else np.nan
    )


    transition_offset = (

        first_predicted_degraded_order
        -
        true_transition_order

        if (
            pd.notna(
                first_predicted_degraded_order
            )
            and
            pd.notna(
                true_transition_order
            )
        )

        else np.nan
    )


    accepted = g[
        g[
            "accepted"
        ]
    ]


    UNIT_ROWS.append({

        "unit":
            unit,

        "n_windows":
            len(g),

        "accuracy":
            g[
                "correct"
            ].mean(),

        "mean_cross_trace_agreement":
            g[
                "agreement"
            ].mean(),

        "within_state_adjacent_stability":
            (
                np.mean(
                    local_same_state
                )
                if local_same_state
                else np.nan
            ),

        "true_transition_order":
            true_transition_order,

        "first_predicted_degraded_order":
            first_predicted_degraded_order,

        "transition_offset_windows":
            transition_offset,

        "coverage":
            g[
                "accepted"
            ].mean(),

        "accepted_accuracy":
            (
                accepted[
                    "correct"
                ].mean()
                if len(
                    accepted
                )
                else np.nan
            ),

        "true_sequence":
            " -> ".join(
                g[
                    "true_label"
                ].astype(str)
            ),

        "prediction_sequence":
            " -> ".join(
                g[
                    "prediction"
                ].astype(str)
            ),

        "disposition_sequence":
            " -> ".join(
                g[
                    "disposition"
                ].astype(str)
            )
    })


TEMP_UNIT_SUMMARY = pd.DataFrame(
    UNIT_ROWS
)


# ============================================================
# Overall summary
# ============================================================

ACCEPTED_TEMP = (
    TEMP_RESULT[
        TEMP_RESULT[
            "accepted"
        ]
    ]
)


TEMP_OVERALL = pd.DataFrame(
    [
        {

            "n_engines":
                TEMP_RESULT[
                    "unit"
                ].nunique(),

            "n_distinct_windows":
                TEMP_RESULT[
                    "instance_id"
                ].nunique(),

            "accuracy":
                TEMP_RESULT[
                    "correct"
                ].mean(),

            "mean_cross_trace_agreement":
                TEMP_RESULT[
                    "agreement"
                ].mean(),

            "within_state_adjacent_stability":
                (
                    np.mean(
                        same_state_adjacent
                    )
                    if same_state_adjacent
                    else np.nan
                ),

            "automatic_coverage":
                TEMP_RESULT[
                    "accepted"
                ].mean(),

            "accepted_accuracy":
                (
                    ACCEPTED_TEMP[
                        "correct"
                    ].mean()
                    if len(
                        ACCEPTED_TEMP
                    )
                    else np.nan
                ),

            "parse_success_rate":
                TEMP_RESULT[
                    "parse_success_rate"
                ].mean(),

            "grounded_rate":
                TEMP_RESULT[
                    "grounded_rate"
                ].mean(),

            "unsupported_rate":
                TEMP_RESULT[
                    "unsupported_rate"
                ].mean(),

            "mean_latency_per_trace_s":
                TEMP_RESULT[
                    "mean_latency_s"
                ].mean(),

            "temporal_input_tokens":
                TEMP_RESULT[
                    "input_tokens"
                ].sum(),

            "temporal_output_tokens":
                TEMP_RESULT[
                    "output_tokens"
                ].sum(),

            "temporal_estimated_cost_usd":
                TEMP_RESULT[
                    "cost_usd"
                ].sum()
        }
    ]
)


TEMP_UNIT_SUMMARY.to_csv(
    RESULTS /
    "nasa_temporal_unit_summary.csv",
    index=False
)

TEMP_OVERALL.to_csv(
    RESULTS /
    "nasa_temporal_overall_summary.csv",
    index=False
)


print(
    "\n"
    +
    "=" * 80
)

print(
    "TEMPORAL WINDOW RESULTS"
)

print(
    "=" * 80
)

display(

    TEMP_RESULT[
        [
            "unit",
            "window_order",
            "window_end",
            "rul_end",
            "true_label",
            "prediction",
            "agreement",
            "verifier_prediction",
            "verification_conflict",
            "disposition",
            "correct"
        ]
    ]
)


print(
    "\nPER-ENGINE TEMPORAL SUMMARY"
)

display(
    TEMP_UNIT_SUMMARY
)


print(
    "\nOVERALL TEMPORAL SUMMARY"
)

display(
    TEMP_OVERALL
)


# ============================================================
# Final persistent API/cost summary
# ============================================================

API_ALL_FINAL, API_SUMMARY_FINAL = (
    persistent_api_summary(
        include_temporal=True
    )
)


print(
    "\n"
    +
    "=" * 80
)

print(
    "FINAL PERSISTENT API SUMMARY"
)

print(
    "=" * 80
)

display(
    API_SUMMARY_FINAL
)


print(
    "\nExpected total persistent traces after temporal run: 8675"
)

print(
    "Actual:",
    len(
        API_ALL_FINAL
    )
)

print(
    "\nZERO additional API calls were made by this summary cell."
)

## 13. Paper figures

In [ ]:
# ============================================================
# FIGURE 2
# NASA FD001 Predictive Performance and Selective Governance
# NO API / NO LLM CALLS
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ------------------------------------------------------------
# 1. FROZEN AUTHORITATIVE RESULTS
# ------------------------------------------------------------

methods = [
    "Logistic\nRegression",
    "SVM-RBF",
    "Random\nForest",
    "Retrieval\nOnly",
    "Prototype\nVerifier",
    "Raw\nCoT-FD6",
    "Accepted\nCoT-FD6"
]

accuracy = np.array([
    0.920,
    0.920,
    0.930,
    0.860,
    0.910,
    0.618,
    0.900
])

accuracy_sd = np.array([
    0.000,
    0.000,
    0.000,
    0.000,
    0.000,
    0.018,
    0.008
])

coverage_mean = 0.660
coverage_sd = 0.020


# ------------------------------------------------------------
# 2. FIGURE STYLE
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9
})

fig, ax = plt.subplots(figsize=(8.4, 4.8))

x = np.arange(len(methods))

bars = ax.bar(
    x,
    accuracy,
    yerr=accuracy_sd,
    capsize=4,
    width=0.68
)

# Distinguish selective accuracy from ordinary predictive accuracy
bars[-1].set_hatch("//")


# ------------------------------------------------------------
# 3. AXES
# ------------------------------------------------------------

ax.set_ylabel("Accuracy")
ax.set_ylim(0.50, 1.02)

ax.set_xticks(x)
ax.set_xticklabels(methods)

ax.set_title(
    "NASA FD001: Predictive Performance and Selective Governance"
)

ax.yaxis.grid(
    True,
    linestyle="--",
    linewidth=0.6,
    alpha=0.45
)

ax.set_axisbelow(True)

# Separate reference/predictive methods from CoT-FD6 results
ax.axvline(
    4.5,
    linestyle=":",
    linewidth=0.9,
    alpha=0.7
)


# ------------------------------------------------------------
# 4. VALUES ABOVE BARS
# ------------------------------------------------------------

for bar, value, sd in zip(bars, accuracy, accuracy_sd):

    xpos = bar.get_x() + bar.get_width() / 2

    if sd > 0:
        label = f"{value:.3f} ± {sd:.3f}"
    else:
        label = f"{value:.3f}"

    ax.text(
        xpos,
        value + 0.018,
        label,
        ha="center",
        va="bottom",
        fontsize=8
    )


# ------------------------------------------------------------
# 5. SELECTIVE GOVERNANCE ANNOTATION
# ------------------------------------------------------------

accepted_bar = bars[-1]

accepted_x = (
    accepted_bar.get_x()
    + accepted_bar.get_width() / 2
)

ax.annotate(
    "Selective accuracy\n"
    f"Coverage = {coverage_mean:.3f} ± {coverage_sd:.3f}",
    xy=(accepted_x, 0.900),
    xytext=(5.35, 0.745),
    ha="center",
    va="center",
    fontsize=8,
    arrowprops=dict(
        arrowstyle="->",
        linewidth=0.8
    )
)

fig.tight_layout()


# ------------------------------------------------------------
# 6. SAVE WITHOUT MATPLOTLIB PDF BACKEND
# ------------------------------------------------------------

OUTPUT_DIR = str(FIGURES)
os.makedirs(OUTPUT_DIR, exist_ok=True)

png_path = os.path.join(
    OUTPUT_DIR,
    "Fig2_NASA_Predictive_Governance.png"
)

jpg_path = os.path.join(
    OUTPUT_DIR,
    "Fig2_NASA_Predictive_Governance.jpg"
)

pdf_path = os.path.join(
    OUTPUT_DIR,
    "Fig2_NASA_Predictive_Governance.pdf"
)

# Save high-resolution master using Matplotlib raster backend
fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Convert the PNG using Pillow.
# This avoids Matplotlib's broken backend_pdf completely.
img = Image.open(png_path).convert("RGB")

img.save(
    jpg_path,
    "JPEG",
    quality=95,
    dpi=(600, 600)
)

img.save(
    pdf_path,
    "PDF",
    resolution=600
)

print("Figure 2 saved successfully:")
print("PNG:", png_path)
print("JPG:", jpg_path)
print("PDF:", pdf_path)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Data
# -----------------------------
# Panel (a): Ablation on NASA FD001 (36-instance subset)
ablation_labels = [
    "A1\nSingle-trace",
    "A2\nMulti-trace",
    "A3\nMulti-trace\n+ verifier",
    "A4\nFull CoT-FD6"
]

acc = np.array([0.241, 0.222, 0.222, 0.657])
f1 = np.array([0.224, 0.200, 0.200, 0.622])

# Only A4 has visible error bars in the figure
acc_sd = np.array([0.000, 0.000, 0.000, 0.032])
f1_sd  = np.array([0.000, 0.000, 0.000, 0.035])

# Coverage shown only for A3 and A4
coverage_x = np.array([2, 3])          # positions of A3 and A4
coverage_y = np.array([0.278, 0.713])

# Panel (b): Retrieval-only vs raw CoT-FD6
comp_labels = ["Synthetic", "NASA FD001"]

retr_acc = np.array([0.980, 0.860])
retr_f1  = np.array([0.979, 0.813])

raw_acc = np.array([0.998, 0.618])
raw_f1  = np.array([0.998, 0.587])

raw_acc_sd = np.array([0.004, 0.018])
raw_f1_sd  = np.array([0.004, 0.015])

# -----------------------------
# Figure layout
# -----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))
fig.suptitle("Ablation and Retrieval Contribution", fontsize=20, y=0.98)

# =============================
# Panel (a)
# =============================
x = np.arange(len(ablation_labels))
w = 0.34

bars1 = ax1.bar(
    x - w/2, acc, w, label="Accuracy",
    yerr=acc_sd, capsize=4
)
bars2 = ax1.bar(
    x + w/2, f1, w, label="Macro-F1",
    yerr=f1_sd, capsize=4
)

ax1.set_title("(a) NASA FD001 ablation on the 36-instance subset", fontsize=16)
ax1.set_ylabel("Predictive score", fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(ablation_labels)
ax1.set_ylim(0, 1.05)
ax1.grid(axis="y", linestyle="--", alpha=0.5)
ax1.legend(loc="upper left", frameon=True)

# Secondary axis for coverage
ax1b = ax1.twinx()
ax1b.plot(
    coverage_x, coverage_y,
    marker="o", linestyle="--", label="Coverage"
)
ax1b.set_ylabel("Automatic coverage", fontsize=14)
ax1b.set_ylim(0, 1.05)
ax1b.legend(loc="upper right", frameon=True)

# Value labels for bars
for i, b in enumerate(bars1):
    h = b.get_height()
    if acc_sd[i] > 0:
        ax1.text(
            b.get_x() + b.get_width()/2, h + 0.025,
            f"{h:.3f} ± {acc_sd[i]:.3f}",
            ha="center", va="bottom", fontsize=10
        )
    else:
        ax1.text(
            b.get_x() + b.get_width()/2, h + 0.015,
            f"{h:.3f}",
            ha="center", va="bottom", fontsize=10
        )

for i, b in enumerate(bars2):
    h = b.get_height()
    if f1_sd[i] > 0:
        ax1.text(
            b.get_x() + b.get_width()/2, h + 0.025,
            f"{h:.3f} ± {f1_sd[i]:.3f}",
            ha="center", va="bottom", fontsize=10
        )
    else:
        ax1.text(
            b.get_x() + b.get_width()/2, h + 0.015,
            f"{h:.3f}",
            ha="center", va="bottom", fontsize=10
        )

# Coverage labels
for xi, yi in zip(coverage_x, coverage_y):
    ax1b.text(
        xi, yi + 0.02, f"{yi:.3f}",
        ha="center", va="bottom", fontsize=10
    )

# N/A labels for coverage under A1 and A2
ax1.text(0, 0.05, "N/A", ha="center", va="bottom", fontsize=11)
ax1.text(1, 0.05, "N/A", ha="center", va="bottom", fontsize=11)

# =============================
# Panel (b)
# =============================
x2 = np.arange(len(comp_labels))
w2 = 0.18

b1 = ax2.bar(x2 - 1.5*w2, retr_acc, w2, label="Retrieval-only Acc.")
b2 = ax2.bar(x2 - 0.5*w2, retr_f1,  w2, label="Retrieval-only F1")
b3 = ax2.bar(x2 + 0.5*w2, raw_acc,  w2, label="Raw CoT-FD6 Acc.", yerr=raw_acc_sd, capsize=4)
b4 = ax2.bar(x2 + 1.5*w2, raw_f1,   w2, label="Raw CoT-FD6 F1",  yerr=raw_f1_sd, capsize=4)

ax2.set_title("(b) Retrieval-only classification versus raw CoT-FD6", fontsize=16)
ax2.set_ylabel("Predictive score", fontsize=14)
ax2.set_xticks(x2)
ax2.set_xticklabels(comp_labels)
ax2.set_ylim(0, 1.14)   # extra headroom to prevent overlap
ax2.grid(axis="y", linestyle="--", alpha=0.5)
ax2.legend(loc="lower left", frameon=True)

# ---- Manual annotation positions to avoid overlap ----
# Synthetic group (index 0)
ax2.text(
    x2[0] - 1.5*w2, retr_acc[0] + 0.025, f"{retr_acc[0]:.3f}",
    ha="center", va="bottom", fontsize=9
)
ax2.text(
    x2[0] - 0.5*w2, retr_f1[0] + 0.025, f"{retr_f1[0]:.3f}",
    ha="center", va="bottom", fontsize=9
)
ax2.text(
    x2[0] + 0.5*w2 - 0.015, raw_acc[0] + 0.040, f"{raw_acc[0]:.3f} ± {raw_acc_sd[0]:.3f}",
    ha="center", va="bottom", fontsize=9
)
ax2.text(
    x2[0] + 1.5*w2 + 0.015, raw_f1[0] + 0.075, f"{raw_f1[0]:.3f} ± {raw_f1_sd[0]:.3f}",
    ha="center", va="bottom", fontsize=9
)

# NASA FD001 group (index 1)
ax2.text(
    x2[1] - 1.5*w2, retr_acc[1] + 0.025, f"{retr_acc[1]:.3f}",
    ha="center", va="bottom", fontsize=9
)
ax2.text(
    x2[1] - 0.5*w2, retr_f1[1] + 0.025, f"{retr_f1[1]:.3f}",
    ha="center", va="bottom", fontsize=9
)
ax2.text(
    x2[1] + 0.5*w2, raw_acc[1] + 0.030, f"{raw_acc[1]:.3f} ± {raw_acc_sd[1]:.3f}",
    ha="center", va="bottom", fontsize=9
)
ax2.text(
    x2[1] + 1.5*w2, raw_f1[1] + 0.030, f"{raw_f1[1]:.3f} ± {raw_f1_sd[1]:.3f}",
    ha="center", va="bottom", fontsize=9
)

# -----------------------------
# Final layout
# -----------------------------
plt.tight_layout(rect=[0, 0, 1, 0.94])
FIGURES.mkdir(parents=True, exist_ok=True)
fig3_path = FIGURES / "Figure3_Ablation_Retrieval_Contribution.png"
fig.savefig(fig3_path, dpi=300, bbox_inches="tight")
print(f"Saved: {fig3_path}")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# FIGURE 4
# Temporal Diagnostic Stability on NASA FD001
#
# Uses the fixed temporal-analysis results already obtained.
# No model/API calls and no experiment rerun are required.
# ============================================================

# -----------------------------
# Data
# -----------------------------
engines = ["Engine 1", "Engine 5", "Engine 11"]

# Five sequential evaluation windows per engine
window_order = np.arange(1, 6)

# Remaining Useful Life (RUL) associated with each selected window
rul = {
    "Engine 1":  [42, 32, 22, 12,  2],
    "Engine 5":  [49, 39, 29, 19,  9],
    "Engine 11": [50, 40, 30, 20, 10],
}

# Retrospective benchmark class:
# Healthy = 0
# Degraded = 1
#
# RUL <= 30 is labelled degraded.
true_state = {
    "Engine 1":  [0, 0, 1, 1, 1],
    "Engine 5":  [0, 0, 1, 1, 1],
    "Engine 11": [0, 0, 1, 1, 1],
}

# Raw modal CoT-FD6 temporal diagnosis
# The model diagnosed all selected windows as degraded.
raw_prediction = {
    "Engine 1":  [1, 1, 1, 1, 1],
    "Engine 5":  [1, 1, 1, 1, 1],
    "Engine 11": [1, 1, 1, 1, 1],
}

# Temporal summary statistics
mean_agreement = {
    "Engine 1":  1.000,
    "Engine 5":  1.000,
    "Engine 11": 0.920,
}

coverage = {
    "Engine 1":  0.800,
    "Engine 5":  1.000,
    "Engine 11": 0.800,
}

accepted_accuracy = {
    "Engine 1":  0.750,
    "Engine 5":  0.600,
    "Engine 11": 0.750,
}

# All engines had:
# true transition at evaluated window 3
# first predicted degraded state at evaluated window 1
# transition offset = 1 - 3 = -2 windows
transition_offset = -2


# -----------------------------
# Plot settings
# -----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11
})

fig, axes = plt.subplots(
    1, 3,
    figsize=(15.5, 5.2),
    sharey=True
)

fig.suptitle(
    "NASA FD001: Temporal Diagnostic Stability",
    fontsize=18,
    y=0.98
)


# -----------------------------
# Draw each engine panel
# -----------------------------
for ax, engine in zip(axes, engines):

    y_true = np.array(true_state[engine], dtype=float)
    y_pred = np.array(raw_prediction[engine], dtype=float)

    # Retrospective benchmark state
    ax.plot(
        window_order,
        y_true,
        marker="o",
        linewidth=2,
        markersize=7,
        label="Retrospective benchmark label"
    )

    # Raw CoT-FD6 modal diagnosis
    ax.plot(
        window_order,
        y_pred,
        marker="s",
        linestyle="--",
        linewidth=2,
        markersize=7,
        label="Raw CoT-FD6 diagnosis"
    )

    # Retrospective Healthy -> Degraded transition occurs
    # between evaluated windows 2 and 3.
    ax.axvline(
        x=2.5,
        linestyle=":",
        linewidth=1.5
    )

    # Annotation for benchmark transition
    ax.text(
        2.55,
        0.48,
        "RUL-based\ntransition",
        rotation=90,
        va="center",
        ha="left",
        fontsize=9
    )

    # X-axis labels: evaluated window + RUL
    tick_labels = [
        f"W{i}\nRUL {r}"
        for i, r in zip(window_order, rul[engine])
    ]

    ax.set_xticks(window_order)
    ax.set_xticklabels(tick_labels)

    # Binary diagnostic-state axis
    ax.set_ylim(-0.18, 1.22)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Healthy", "Degraded"])

    ax.set_xlim(0.7, 5.3)

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.45
    )

    # Engine-specific summary
    ax.set_title(
        f"{engine}\n"
        f"Agreement = {mean_agreement[engine]:.3f}, "
        f"Coverage = {coverage[engine]:.3f}",
        fontsize=12
    )

    # Transition-offset annotation
    ax.text(
        0.98,
        0.06,
        f"Transition offset = {transition_offset} windows",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9
    )


# -----------------------------
# Shared labels
# -----------------------------
axes[0].set_ylabel(
    "Diagnostic state",
    fontsize=13
)

for ax in axes:
    ax.set_xlabel(
        "Sequential evaluated window",
        fontsize=11
    )


# -----------------------------
# One shared legend
# -----------------------------
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=2,
    frameon=True,
    bbox_to_anchor=(0.5, -0.01)
)


# -----------------------------
# Layout
# -----------------------------
plt.tight_layout(
    rect=[0, 0.10, 1, 0.91],
    w_pad=2.0
)


# -----------------------------
# Save high-resolution image
# -----------------------------
FIGURES.mkdir(parents=True, exist_ok=True)
png_path = FIGURES / "Figure4_Temporal_Diagnostic_Stability.png"

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

print(f"Saved: {png_path}")

plt.show()

## 14. Result-file map

| Manuscript analysis | Primary output |
|---|---|
| Like-for-like conventional baselines | `results/baseline_metrics.csv` |
| Main repeated LLM predictions | `results/cotfd6_main_predictions.csv` |
| Main raw LLM traces | `results/cotfd6_main_raw_traces.csv` |
| Main metrics by repeat | `results/cotfd6_main_metrics_by_repeat.csv` |
| Main mean ± SD | `results/cotfd6_main_metrics_mean_sd.csv` |
| Trace-count sensitivity | `results/trace_sensitivity_predictions.csv` |
| Ablation | `results/ablation_predictions.csv` |
| Retrieval-only baseline | `results/memory_only_full100_predictions.csv` |
| RF-SHAP | `results/rf_shap_top_features.csv` |
| Explanation grounding | `results/explanation_grounding_summary.csv` |
| NASA confusion counts | `results/nasa_raw_confusion_counts.csv` |
| FN/FP sensitivity | `results/nasa_cost_sensitivity.csv` |
| API tokens/latency/cost | `results/api_cost_latency_summary.csv` |
| Temporal stability | `results/nasa_temporal_diagnostic_stability.csv` |
| Frozen configuration | `results/experiment_config.json` |

Publish the authoritative result CSVs produced by the reported run with the release/Zenodo archive. Do not regenerate manuscript numbers from exploratory pilot cells.